# Two-Stage Industrial Demand Response — Pyomo MILP

This notebook schedules **two machines** — a **Chiron 800 CNC milling machine** and an
**HML MP 50 1 VK chip press** — to **minimize day-ahead electricity procurement cost**
while meeting delivery deadlines, respecting storage capacities, and exploiting
**load flexibility** (time shifting and eco mode).

This is **step 1 of 3** of a full-year study (study year **2024**). This step builds the
refactored model and the data layer:

1. removes the on-site PV array from the study (see rationale below),
2. loads day-ahead prices **timezone-correctly** (UTC → Europe/Berlin),
3. loads the business-day calendar,
4. generates a deterministic **synthetic order book** for all 2024 business days,
5. runs a **year-wide analytical feasibility pre-check** and auto-tunes the capacity
   parameters until every business day passes,
6. persists the full run configuration, and
7. still solves the single-day model for **2024-03-18** as a smoke test.

The rolling-horizon year run, the business case, and the paper figures are **not** part of
this step.

> **Disclaimer.** This is a **fictional but plausible** industrial process constructed to
> demonstrate energy flexibility in manufacturing — **not a real electrical-engineering process
> model**. Quantities, deadlines, and stock levels are chosen for illustration.

## Process

1. **Milling stage (Chiron 800):** mills aluminum profiles into power-module housings.
   One run produces **100 housings** (intermediate product). Two modes: **normal** (fixed load
   template) and **eco** (every load point ×0.75, duration ×1.25, rounded to the 15-min grid).
2. **Press stage (chip press):** presses/bonds multilayer substrates into modules under heat and
   pressure. One run **consumes 40 housings and produces 40 finished power modules** with a
   **fixed, non-deformable** load template — only **shiftable in time**.

## Assumptions

- **Operating window:** 08:00–22:00 **local time**, **15-min resolution → 56 time steps**.
- **Template aggregation:** the 1-min templates are mapped onto the model grid via **mean per
  15-min bin** (average power per bin, in kW).
- **Booking convention:** production (housings/modules) is credited at **session end**; the press
  books its **housing consumption at start**.
- **No PV:** electricity is procured entirely from the grid at the day-ahead price. Cost applies
  to the full machine load at each slot.
- **1:1 assumption:** 40 housings consumed → 40 modules produced per press run.
- **Eco energy:** ×0.75 power × ×1.25 duration ⇒ ~0.94× the energy of a normal run (lower peak,
  longer occupancy — the flexibility trade-off).

## Why PV was removed

PV is dropped from the study, and this is more than a simplification. **5.2 % of all 15-min slots
in 2024 have negative day-ahead prices.** The old formulation modelled grid draw as
`grid[t] ≥ load[t] − PV[t]` with only a lower bound on `grid[t]`; at a negative price the solver
could drive `grid[t]` arbitrarily high to *collect* money, so the problem would have been
**unbounded**. Without PV the objective is the machine load valued directly at the price, negative
prices are handled correctly, and they simply become **attractive production windows**.

## Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import pyomo.environ as pyo
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from datetime import time, date, timedelta

pio.renderers.default = "notebook"   # embed plotly.js into the notebook output
RESDIR = "results"; os.makedirs(RESDIR, exist_ok=True)   # RUN_DIR (per run) is created in the Parameters section

# unified color scheme (consistent with the EDA)
C = {"mill": "#1f5fbf", "press": "#e8720c", "price": "#6f6f6f",
     "housing": "#1f5fbf", "module": "#e8720c"}

## Parameters

Central, easy-to-edit parameters. **Plain Python numbers** (no Pyomo objects).

The demand is no longer a hand-picked pickup dict — it comes from the **synthetic order book**
(generated further down). The capacity parameters below are **starting values**; the year-wide
feasibility pre-check auto-tunes the tunable ones until every 2024 business day passes.

**Per-day vs. per-window caps.** The run-count caps are defined **per production day**. Step 2 of
the study will model a **two-day rolling window** (112 slots), so the window-level caps are
derived here as `2 × per-day`. The single-day smoke test in this notebook uses the per-day caps.

> **Session durations are not parameters here.** The normal-mill and press durations are
> **derived from the template CSV lengths** (see the Data section) so they can never desync from
> the load shape. The only duration knob is the **eco time-stretch** factor.

In [ ]:
# --- Study scope -----------------------------------------------------------
STUDY_YEAR = 2024          # the study covers 2024 only
SEED       = 187           # single random seed for all randomness (order book)

# --- Production / consumption ----------------------------------------------
HOUSINGS_PER_MILL_RUN            = 100   # housings produced per mill run
MODULES_PER_PRESS_RUN            = 40    # modules produced per press run
HOUSINGS_CONSUMED_PER_PRESS_RUN  = 40    # housings consumed per press run (1:1)

# --- Initial stock / storage (TUNABLE) -------------------------------------
INITIAL_STOCK_HOUSINGS    = 60           # >=40 -> press can start at 08:00
INITIAL_STOCK_MODULES     = 30
STORAGE_CAPACITY_HOUSINGS = 300          # cap on intermediate product
STORAGE_CAPACITY_MODULES  = 450          # cap on finished product

# --- Eco mode (the only real duration parameter: time stretch) -------------
ECO_MODE_POWER_SCALING = 0.75
ECO_MODE_TIME_STRETCH  = 1.25
# Note: normal-mill and press durations are NOT free parameters — they are
#       derived from the aggregated template lengths (Data section).

# --- Time horizon ----------------------------------------------------------
WORKDAY_START = "08:00"
WORKDAY_END   = "22:00"
STEP_MIN      = 15
N_T           = 56                       # (22:00-08:00)/15min = 56 slots
STEP_H        = STEP_MIN / 60.0          # 0.25 h per slot

# --- Session upper bounds (solver degrees of freedom), PER DAY (TUNABLE) ----
MAX_MILL_RUNS_PER_DAY  = 4
MAX_PRESS_RUNS_PER_DAY = 10
# Window-level caps for the two-day rolling window in step 2 (derived, do not edit):
MAX_MILL_RUNS_PER_WINDOW  = 2 * MAX_MILL_RUNS_PER_DAY
MAX_PRESS_RUNS_PER_WINDOW = 2 * MAX_PRESS_RUNS_PER_DAY

# --- Run labelling (for the output folder name) ----------------------------
MODEL_DAY = "2024-03-18"   # single day used for the smoke-test solve
UTIL_NOTE = "smoke-test"   # free label per run

# --- Output folder of this run (all plots AND CSVs land here) ---------------
_h0, _h1 = int(WORKDAY_START[:2]), int(WORKDAY_END[:2])
RUN_DIR = os.path.join(RESDIR, f"run_{MODEL_DAY}_{_h0}-{_h1}_{UTIL_NOTE}")
os.makedirs(RUN_DIR, exist_ok=True)
print("Horizon:", N_T, "slots of", STEP_MIN, "min | Eco: power x",
      ECO_MODE_POWER_SCALING, ", time x", ECO_MODE_TIME_STRETCH)
print("Per-day caps  -> mill:", MAX_MILL_RUNS_PER_DAY, "| press:", MAX_PRESS_RUNS_PER_DAY)
print("Per-window caps -> mill:", MAX_MILL_RUNS_PER_WINDOW, "| press:", MAX_PRESS_RUNS_PER_WINDOW)
print("Run folder:", RUN_DIR)

## Data — machine templates

Load the two 1-min machine templates from `templates/` and aggregate them onto the 15-min model
grid. The day-ahead price data layer and the calendar follow in their own sections.

In [ ]:
def resample_1min_to_15(series_1min):
    "Mean power per 15-min bin (kW)."
    v = np.asarray(series_1min, dtype=float)
    n = len(v) // STEP_MIN
    return v[:n * STEP_MIN].reshape(n, STEP_MIN).mean(axis=1)

# Raw templates (1-min)
mill_1min  = pd.read_csv("templates/chiron_session6_min230-380_1min.csv")["P_kW"].to_numpy(float)
press_1min = pd.read_csv("templates/chippress_session4_window_1min.csv")["P_kW"].to_numpy(float)

# Normal mill + press: aggregate directly to 15-min
mill_normal = resample_1min_to_15(mill_1min)   # 10 slots
press_tpl   = resample_1min_to_15(press_1min)  # 5 slots

# Eco mill: power x0.75, time x1.25 (interpolate), then 15-min bins (last bin partial)
L = int(round(len(mill_1min) * ECO_MODE_TIME_STRETCH))                 # 188 min
ECO_SLOTS = int(np.ceil(L / STEP_MIN))                                 # 13 slots (covers 187.5 min)
eco_1min = ECO_MODE_POWER_SCALING * np.interp(
    np.linspace(0, len(mill_1min) - 1, L), np.arange(len(mill_1min)), mill_1min)
mill_eco = np.array([eco_1min[i*STEP_MIN:(i+1)*STEP_MIN].mean() for i in range(ECO_SLOTS)])

mill_tpl = {"normal": mill_normal, "eco": mill_eco}

# --- Session durations in slots DERIVED FROM THE TEMPLATE LENGTHS ----------
# (normal mill & press = length of the aggregated templates; eco = ECO_SLOTS)
DUR = {"normal": len(mill_normal), "eco": ECO_SLOTS}
PRESS_DUR = len(press_tpl)
print(f"Derived slot durations -> mill normal: {DUR['normal']} | mill eco: {DUR['eco']} | "
      f"press: {PRESS_DUR}  ({STEP_MIN} min each)")
print("Mill normal (kW/slot):", np.round(mill_normal, 2))
print("Mill eco    (kW/slot):", np.round(mill_eco, 2))
print("Press       (kW/slot):", np.round(press_tpl, 2))
print(f"Energy/run: mill-normal {mill_normal.sum()*STEP_H:.2f} kWh | "
      f"eco {mill_eco.sum()*STEP_H:.2f} kWh | press {press_tpl.sum()*STEP_H:.2f} kWh")

## Calendar & time helpers

Load the business-day calendar (`data/business_days.csv`, one row per calendar day with a
`werktag` flag) and define the slot ↔ clock-time helpers used by the price layer, the order book
and the pre-check. Business days are the rows with `werktag == 1`; there is **no production and no
pickup on non-business days**, inventory simply carries over.

In [ ]:
# --- Slot <-> clock-time helpers (operating window starts at WORKDAY_START) --
def slot_to_hhmm(t):
    mins = 8 * 60 + t * STEP_MIN
    return f"{mins // 60:02d}:{mins % 60:02d}"

def hhmm_to_slot(h):
    hh, mm = map(int, h.split(":"))
    return (hh * 60 + mm - 8 * 60) // STEP_MIN

def pickup_slot(h):
    # A pickup at WORKDAY_END (22:00) maps to the LAST slot of the day (slot 55, i.e. 21:45),
    # because 22:00 is the window boundary. It is treated as an end-of-day deadline: everything
    # must be finished by then. This preserves the existing min(hhmm_to_slot(k), N_T-1) behaviour.
    return min(hhmm_to_slot(h), N_T - 1)

# --- Business-day calendar -------------------------------------------------
_cal = pd.read_csv("data/business_days.csv", parse_dates=["date"])
_cal["date"] = _cal["date"].dt.date
_cal = _cal[pd.DatetimeIndex(_cal["date"]).year == STUDY_YEAR].sort_values("date").reset_index(drop=True)

BUSINESS_DATES = [d for d, w in zip(_cal["date"], _cal["werktag"]) if int(w) == 1]
_business_set  = set(BUSINESS_DATES)

def is_business_day(d):
    "True if date d is a business day in the study calendar."
    return d in _business_set

def next_business_day(d):
    "Next date strictly after d flagged as a business day (needed for the look-ahead day in step 2)."
    nxt = d + timedelta(days=1)
    last = BUSINESS_DATES[-1]
    while nxt <= last:
        if nxt in _business_set:
            return nxt
        nxt += timedelta(days=1)
    return None   # d is on/after the last business day of the calendar

assert len(BUSINESS_DATES) == 252, f"expected 252 business days in {STUDY_YEAR}, got {len(BUSINESS_DATES)}"
print(f"Business days in {STUDY_YEAR}: {len(BUSINESS_DATES)} "
      f"(first {BUSINESS_DATES[0]}, last {BUSINESS_DATES[-1]})")
print("Example look-ahead: next business day after 2024-03-15 (Fri) is",
      next_business_day(date(2024, 3, 15)))

### Template comparison: 1-min vs. 15-min (aggregation)

In [ ]:
def step_from_15(tpl):
    "15-min template as minute-x for a step plot."
    x, y = [], []
    for i, p in enumerate(tpl):
        x += [i*STEP_MIN, (i+1)*STEP_MIN]; y += [p, p]
    return x, y

fig = make_subplots(rows=1, cols=3, subplot_titles=("Mill normal", "Mill eco", "Press"))
for col, (raw, tpl, dur) in enumerate([
        (mill_1min, mill_normal, DUR["normal"]),
        (eco_1min,  mill_eco,    DUR["eco"]),
        (press_1min, press_tpl,  PRESS_DUR)], start=1):
    fig.add_trace(go.Scatter(y=raw, x=np.arange(len(raw)), mode="lines",
                             line=dict(color="#1f5fbf", width=1), name="1-min",
                             showlegend=(col == 1)), row=1, col=col)
    sx, sy = step_from_15(tpl)
    fig.add_trace(go.Scatter(x=sx, y=sy, mode="lines", line=dict(color="#c0392b", width=2.5),
                             name="15-min mean", showlegend=(col == 1)), row=1, col=col)
    fig.update_xaxes(title_text="min", row=1, col=col)
fig.update_yaxes(title_text="kW", row=1, col=1)
fig.update_layout(height=360, width=1050, title="Templates: 1-min raw profile vs. 15-min aggregation",
                  template="plotly_white")
fig.write_image(f"{RUN_DIR}/opt_templates_1min_vs_15min.png", scale=2)
fig.show()

## Day-ahead prices (timezone-correct)

The price file `data/electricity_prices_15min_de_2018_2024.csv` is stored in **UTC**
(`utc_datetime`), whereas the operating window (08:00–22:00) and the calendar are in **local
time** (`Europe/Berlin`). The previous code cut the window with a fixed index offset
(`s0 = 8*60 // STEP_MIN`), which silently assumed the daily array started at *local* midnight —
off by one hour between winter (CET, UTC+1) and summer (CEST, UTC+2).

We fix this properly: parse `utc_datetime` as timezone-aware UTC, convert to `Europe/Berlin`,
derive `local_date` / `local_time`, and select the window **by local time** (`>= 08:00` and
`< 22:00`) — never by integer index arithmetic. Every 2024 business day must then yield exactly
**56** price slots.

Both 2024 DST transition days (2024-03-31 and 2024-10-27) fall on **Sundays** and are therefore
excluded by the business-day calendar anyway — but the conversion must still be timezone-aware,
because the UTC offset of the 08:00 local boundary differs between summer and winter
(08:00 local = 07:00 UTC under CET, 06:00 UTC under CEST).

In [ ]:
PRICE_FILE = "data/electricity_prices_15min_de_2018_2024.csv"

# 1) parse UTC timezone-aware, 2) convert to Europe/Berlin, 3) derive local date/time columns
prices = pd.read_csv(PRICE_FILE)
prices["utc_datetime"] = pd.to_datetime(prices["utc_datetime"], utc=True)
prices["local_dt"]   = prices["utc_datetime"].dt.tz_convert("Europe/Berlin")
prices["local_date"] = prices["local_dt"].dt.date
prices["local_time"] = prices["local_dt"].dt.time

# --- Verified facts about the price file (assertions, not re-derived) -------
# (computed on the UTC calendar year, matching the documented statistics)
_p2024 = prices.loc[prices["utc_datetime"].dt.year == STUDY_YEAR, "price_eur_per_mwh"]
assert len(_p2024) == 35136,            f"2024 must have 35136 slots, got {len(_p2024)}"
assert int(_p2024.isna().sum()) == 0,   "2024 price series must have zero missing values"
assert np.isclose(_p2024.mean(), 78.51, atol=0.01), f"mean {_p2024.mean():.2f} != 78.51"
assert np.isclose(_p2024.min(), -135.45, atol=0.01), f"min {_p2024.min():.2f} != -135.45"
assert np.isclose(_p2024.max(),  936.28, atol=0.01), f"max {_p2024.max():.2f} != 936.28"
assert int((_p2024 < 0).sum()) == 1828, f"negative slots {(_p2024<0).sum()} != 1828"
print(f"Price file OK: {len(_p2024)} slots in {STUDY_YEAR}, mean {_p2024.mean():.2f}, "
      f"min {_p2024.min():.2f}, max {_p2024.max():.2f}, "
      f"negative {(_p2024<0).sum()} ({(_p2024<0).mean()*100:.1f} %)")

# --- Operating-window selection BY LOCAL TIME (never by index arithmetic) ---
WINDOW_START, WINDOW_END = time(8, 0), time(22, 0)   # 08:00 <= local_time < 22:00
_win = prices[(prices["local_time"] >= WINDOW_START) & (prices["local_time"] < WINDOW_END)] \
          .sort_values("local_dt")
PRICE_BY_DATE = {d: g["price_eur_per_mwh"].to_numpy(float)
                 for d, g in _win.groupby("local_date")}

def prices_for_date(d):
    "56-element day-ahead price vector (EUR/MWh) for the operating window of local date d."
    arr = PRICE_BY_DATE.get(d)
    if arr is None or len(arr) != N_T:
        raise KeyError(f"no complete {N_T}-slot price window for {d}")
    return arr

# Assertion: every 2024 business day yields exactly 56 price slots
_bad = [d for d in BUSINESS_DATES if len(PRICE_BY_DATE.get(d, [])) != N_T]
assert not _bad, f"{len(_bad)} business day(s) without exactly {N_T} price slots, e.g. {_bad[:3]}"
print(f"All {len(BUSINESS_DATES)} business days yield exactly {N_T} price slots (timezone-correct).")

## Synthetic order book (per-slot Bernoulli)

Deterministic order book (seed 187, `numpy.default_rng`). For **each** `(business day, time)` pair
an independent **Bernoulli(p)** decides whether a pickup exists — so at most one pickup per
`(date, time)` holds **by construction** (no filtering, no rejection sampling, no redraws):

```
for each business day D in 2024:
    for each time T in {"15:00", "22:00"}:
        with probability p:
            quantity ~ Normal(mu, sigma), truncate to [80,190], round to nearest 10, re-clip
            emit (D, T, quantity)
```

The tunable knobs (`OB_P`, `OB_MU`, `OB_SIGMA`, `OB_TRUNC`, `OB_ROUND_TO`, `OB_TIMES`) sit at the
top of the cell. The earlier draw-a-count-then-drop-collisions generator capped the distribution at
two pickups/day and pushed utilisation down to ~46 %; the per-slot Bernoulli lifts mean daily
demand to ~270 modules (~67.5 % of the `252 * MAX_PRESS_RUNS_PER_DAY * MODULES_PER_PRESS_RUN`
ceiling) with a hard maximum of 380 modules/day (two pickups of 190). The canonical file is
`data/orderbook_2024.csv`; its meta records the full generator spec. A pickup at `"22:00"` still
maps to the **last slot of the day** (slot 55) via `pickup_slot`.

In [ ]:
OB_FILE      = "data/orderbook_2024.csv"          # the single canonical order book
OB_META_FILE = "data/orderbook_2024_meta.json"

# --- Named generator parameters (iterate on these; the logic below stays fixed) ---
OB_P        = 0.90            # probability a (business day, time) slot carries a pickup
OB_MU       = 150            # pickup quantity mean
OB_SIGMA    = 30             # pickup quantity std
OB_TRUNC    = (80, 190)      # quantity truncation bounds (also re-applied AFTER rounding)
OB_ROUND_TO = 10             # round quantity to nearest multiple
OB_TIMES    = ["15:00", "22:00"]

OB_DIST = {"p": OB_P, "mu": OB_MU, "sigma": OB_SIGMA, "trunc": list(OB_TRUNC),
           "round_to": OB_ROUND_TO, "times": OB_TIMES,
           "rule": "per (business day, time) independent Bernoulli(p); at most one pickup per (date,time) by construction"}

def generate_orderbook():
    "Independent Bernoulli(p) per (business day, time). Uniqueness of (date,time) holds BY CONSTRUCTION."
    rng = np.random.default_rng(SEED)
    lo, hi = OB_TRUNC
    rows = []
    for d in BUSINESS_DATES:
        for t in OB_TIMES:
            if rng.random() < OB_P:
                q = float(np.clip(rng.normal(OB_MU, OB_SIGMA), lo, hi))   # truncate
                q = int(np.clip(round(q / OB_ROUND_TO) * OB_ROUND_TO, lo, hi))  # round, then re-clip
                rows.append({"date": str(d), "time": t, "quantity": q})
    return pd.DataFrame(rows, columns=["date", "time", "quantity"])

# Deterministic (seed 187): always regenerate and write -> the CSV never drifts from OB_DIST.
orderbook = generate_orderbook()
orderbook.to_csv(OB_FILE, index=False)
orderbook["date"] = pd.to_datetime(orderbook["date"]).dt.date
orderbook["slot"] = orderbook["time"].map(pickup_slot)
# uniqueness is guaranteed by the one-draw-per-(day,time) structure; assert it holds
assert orderbook.groupby(["date", "time"]).size().max() == 1, "(date,time) not unique"
assert not orderbook.duplicated(subset=["date", "time"]).any(), "duplicate (date,time) rows"

# --- Realised summary statistics (for the paper) ---------------------------
_daily = orderbook.groupby("date")["quantity"].agg(["count", "sum"])
_capacity = len(BUSINESS_DATES) * MAX_PRESS_RUNS_PER_DAY * MODULES_PER_PRESS_RUN   # 252 * 10 * 40
OB_STATS = {
    "n_business_days": len(BUSINESS_DATES),
    "n_pickups": int(len(orderbook)),
    "mean_pickups_per_day": round(len(orderbook) / len(BUSINESS_DATES), 3),
    "qty_mean": round(float(orderbook["quantity"].mean()), 2),
    "qty_median": float(orderbook["quantity"].median()),
    "qty_max": int(orderbook["quantity"].max()),
    "daily_total_mean": round(float(_daily["sum"].mean()), 2),
    "daily_total_max": int(_daily["sum"].max()),
    "annual_total": int(orderbook["quantity"].sum()),
    "utilisation_pct": round(100.0 * orderbook["quantity"].sum() / _capacity, 2),
    "zero_pickup_days": int(len(BUSINESS_DATES) - orderbook["date"].nunique()),
    "one_pickup_days": int((_daily["count"] == 1).sum()),
    "n_at_1500": int((orderbook["time"] == "15:00").sum()),
    "n_at_2200": int((orderbook["time"] == "22:00").sum()),
}
with open(OB_META_FILE, "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, "study_year": STUDY_YEAR,
               "generator": "numpy.random.default_rng, per-(day,time) Bernoulli",
               "order_book_file": OB_FILE, "distribution": OB_DIST, "stats": OB_STATS}, f, indent=2)

print(f"Generated order book -> {OB_FILE}  (unique (date,time) by construction)")
print("\nOrder book summary")
print(f"  business days              : {OB_STATS['n_business_days']}")
print(f"  pickups (total)            : {OB_STATS['n_pickups']}")
print(f"  mean pickups / day         : {OB_STATS['mean_pickups_per_day']}")
print(f"  quantity mean/median/max   : {OB_STATS['qty_mean']} / {OB_STATS['qty_median']:.0f} / {OB_STATS['qty_max']}")
print(f"  daily total mean/max       : {OB_STATS['daily_total_mean']} / {OB_STATS['daily_total_max']}")
print(f"  annual total               : {OB_STATS['annual_total']}")
print(f"  utilisation                : {OB_STATS['utilisation_pct']} %  of {_capacity} module/yr ceiling")
print(f"  zero-pickup / one-pickup d : {OB_STATS['zero_pickup_days']} ({100*OB_STATS['zero_pickup_days']/OB_STATS['n_business_days']:.1f} %)"
      f" / {OB_STATS['one_pickup_days']} ({100*OB_STATS['one_pickup_days']/OB_STATS['n_business_days']:.1f} %)")
print(f"  split 15:00 / 22:00        : {OB_STATS['n_at_1500']} / {OB_STATS['n_at_2200']}")

## Year-wide feasibility pre-check & auto-tuning

Before any optimisation, run an analytical feasibility check over **all** 2024 business days.

> **Scope, stated honestly.** These are **necessary conditions, not sufficient ones.** The
> definitive test is the actual rolling-horizon year run in step 2. The goal here is to catch the
> *structural* violations cheaply.

**Checks (per day and cumulatively):**

1. **Single-slot module storage.** Pickups landing on the same slot sum up and must all sit in
   inventory at that slot; flag any slot whose summed pickup exceeds the module cap. *This is the
   expected failure mode:* three pickups of up to 200 on one slot demand up to 600 modules, above
   the starting cap of 450.
2. **Daily module throughput.** Daily demand vs. `cap + MAX_PRESS_RUNS_PER_DAY × 40`, i.e. same-day
   production plus the most that can be carried in (opening stock is bounded by the cap).
3. **Housing supply.** Same-day press runs (after drawing down carried module stock) vs. mill
   output plus carried housings.
4. **Time fit.** `runs × duration_in_slots ≤ 56` per machine per day.
5. **Cumulative balance.** For every day *d*, cumulative demand through *d* must not exceed initial
   stock plus cumulative maximum production through *d* (catches sustained overload).

**Auto-tuning.** Tunable: `STORAGE_CAPACITY_HOUSINGS`, `STORAGE_CAPACITY_MODULES`,
`MAX_MILL_RUNS_PER_DAY`, `MAX_PRESS_RUNS_PER_DAY`, `INITIAL_STOCK_HOUSINGS`,
`INITIAL_STOCK_MODULES`. Never tuned: run durations, per-run yields, the operating window, the
order book, the seed, the price data. Storage rises in steps of 50, run counts in steps of 1,
storage is preferred over run counts, and each adjustment is logged with its reason and triggering
date. Physical ceilings (**5 mill runs**, **11 press runs** per 56-slot day) are respected — if the
checks still fail at the ceiling the tuner stops and reports rather than overshooting.

In [ ]:
# Physical per-day ceilings from durations (fixed, never tuned)
MILL_RUNS_CEILING  = N_T // DUR["normal"]   # 56 // 10 = 5
PRESS_RUNS_CEILING = N_T // PRESS_DUR       # 56 //  5 = 11

# Per-day demand aggregates from the order book (non-business days -> zero demand)
_ob_by_date = {}
for _d, _g in orderbook.groupby("date"):
    _ss = _g.groupby("slot")["quantity"].sum().astype(int).to_dict()
    _ob_by_date[_d] = {"slot_sum": _ss, "demand": int(_g["quantity"].sum()),
                       "max_slot": int(max(_ss.values()))}

def year_precheck(P):
    "Return a list of necessary-condition violations across all business days for parameter set P."
    v = []
    cap_mod, cap_hou = P["STORAGE_CAPACITY_MODULES"], P["STORAGE_CAPACITY_HOUSINGS"]
    max_press, max_mill = P["MAX_PRESS_RUNS_PER_DAY"], P["MAX_MILL_RUNS_PER_DAY"]
    init_mod, init_hou = P["INITIAL_STOCK_MODULES"], P["INITIAL_STOCK_HOUSINGS"]
    max_daily_modules  = max_press * MODULES_PER_PRESS_RUN
    max_daily_housings = max_mill  * HOUSINGS_PER_MILL_RUN
    cum_demand = cum_max_mod = cum_hcons = cum_max_hou = 0
    for d in BUSINESS_DATES:
        info = _ob_by_date.get(d, {"slot_sum": {}, "demand": 0, "max_slot": 0})
        demand, max_slot = info["demand"], info["max_slot"]
        # (1) single-slot module storage
        if max_slot > cap_mod:
            v.append({"check": "single_slot_module_storage", "date": d, "need": max_slot,
                      "have": cap_mod, "param": "STORAGE_CAPACITY_MODULES"})
        # (2) daily module throughput (same-day production + carried stock up to cap)
        if demand > cap_mod + max_daily_modules:
            v.append({"check": "daily_module_throughput", "date": d, "need": demand,
                      "have": cap_mod + max_daily_modules, "param": "STORAGE_CAPACITY_MODULES"})
        # same-day press runs needed after drawing down carried module stock (opening <= cap)
        same_day_press = int(np.ceil(max(0, demand - cap_mod) / MODULES_PER_PRESS_RUN))
        need_hou = same_day_press * HOUSINGS_CONSUMED_PER_PRESS_RUN
        # (3) housing supply for those same-day press runs
        if need_hou > cap_hou + max_daily_housings:
            v.append({"check": "housing_supply", "date": d, "need": need_hou,
                      "have": cap_hou + max_daily_housings, "param": "STORAGE_CAPACITY_HOUSINGS"})
        # (4) time fit per machine per day
        if same_day_press * PRESS_DUR > N_T:
            v.append({"check": "time_fit_press", "date": d, "need": same_day_press * PRESS_DUR,
                      "have": N_T, "param": None})
        same_day_mill = int(np.ceil(need_hou / HOUSINGS_PER_MILL_RUN))
        if same_day_mill * DUR["normal"] > N_T:
            v.append({"check": "time_fit_mill", "date": d, "need": same_day_mill * DUR["normal"],
                      "have": N_T, "param": None})
        # (5) cumulative balance
        cum_demand += demand; cum_max_mod += max_daily_modules
        if cum_demand > init_mod + cum_max_mod:
            v.append({"check": "cumulative_module_balance", "date": d, "need": cum_demand,
                      "have": init_mod + cum_max_mod, "param": "MAX_PRESS_RUNS_PER_DAY"})
        cum_hcons += int(np.ceil(demand / MODULES_PER_PRESS_RUN)) * HOUSINGS_CONSUMED_PER_PRESS_RUN
        cum_max_hou += max_daily_housings
        if cum_hcons > init_hou + cum_max_hou:
            v.append({"check": "cumulative_housing_balance", "date": d, "need": cum_hcons,
                      "have": init_hou + cum_max_hou, "param": "MAX_MILL_RUNS_PER_DAY"})
    return v

# Tuning order (Prompt 1b): the higher-utilisation demand is throughput-bound, not storage-bound,
# so escalate RUN COUNTS first (mill 4->5, then press 10->11, up to the physical ceilings) and only
# then storage in steps of 50. (Step 1's storage-first order suited the old storage-bound
# single-slot regime.) Time-fit violations involve fixed run durations and are not tunable.
_PRIORITY = ["single_slot_module_storage", "daily_module_throughput", "housing_supply",
             "cumulative_module_balance", "cumulative_housing_balance",
             "time_fit_press", "time_fit_mill"]
_MODULE_CHECKS = {"single_slot_module_storage", "daily_module_throughput", "cumulative_module_balance"}

TUNABLE = ["STORAGE_CAPACITY_HOUSINGS", "STORAGE_CAPACITY_MODULES",
           "MAX_MILL_RUNS_PER_DAY", "MAX_PRESS_RUNS_PER_DAY",
           "INITIAL_STOCK_HOUSINGS", "INITIAL_STOCK_MODULES"]
_params = {k: globals()[k] for k in TUNABLE}
_before = dict(_params)
tuning_log = []

for _it in range(500):
    viols = year_precheck(_params)
    if not viols:
        break
    kinds = {v["check"] for v in viols}
    trig = sorted(viols, key=lambda v: _PRIORITY.index(v["check"]))[0]
    if kinds <= {"time_fit_press", "time_fit_mill"}:
        print(f"STOP: only time-fit violations remain (e.g. {trig['check']} on {trig['date']}) "
              f"- run durations are fixed and not tunable.")
        break
    # escalation ladder: mill runs -> press runs -> storage (+50)
    if _params["MAX_MILL_RUNS_PER_DAY"] < MILL_RUNS_CEILING:
        p = "MAX_MILL_RUNS_PER_DAY"; old = _params[p]; _params[p] = old + 1
    elif _params["MAX_PRESS_RUNS_PER_DAY"] < PRESS_RUNS_CEILING:
        p = "MAX_PRESS_RUNS_PER_DAY"; old = _params[p]; _params[p] = old + 1
    else:
        p = "STORAGE_CAPACITY_MODULES" if (kinds & _MODULE_CHECKS) else "STORAGE_CAPACITY_HOUSINGS"
        old = _params[p]; _params[p] = old + 50
    tuning_log.append({"param": p, "old": old, "new": _params[p], "reason": trig["check"],
                       "trigger_date": str(trig["date"]), "need": trig["need"], "have_before": trig["have"]})
    print(f"[tune] {p}: {old} -> {_params[p]}  (reason: {trig['check']}, trigger {trig['date']}, "
          f"need {trig['need']} > {trig['have']})")

_final_viols = year_precheck(_params)
_n_days_ok = len(BUSINESS_DATES) - len({v["date"] for v in _final_viols})

# write tuned values back into the global parameters (and re-derive window caps)
for k in TUNABLE:
    globals()[k] = _params[k]
MAX_MILL_RUNS_PER_WINDOW  = 2 * MAX_MILL_RUNS_PER_DAY
MAX_PRESS_RUNS_PER_WINDOW = 2 * MAX_PRESS_RUNS_PER_DAY

# --- before/after report ---------------------------------------------------
print("\nAuto-tuning result")
print(f"  iterations           : {len(tuning_log)}")
print(f"  business days PASS    : {_n_days_ok} / {len(BUSINESS_DATES)}")
print(f"\n{'parameter':28s}{'before':>10s}{'after':>10s}   changed")
for k in TUNABLE:
    ch = "<-- tuned" if _before[k] != _params[k] else ""
    print(f"  {k:26s}{_before[k]:>10}{_params[k]:>10}   {ch}")

if not _final_viols:
    print("\nPASS: every 2024 business day satisfies the necessary feasibility conditions.")
else:
    print(f"\nFAIL: {len(_final_viols)} residual violation(s), e.g. {_final_viols[:3]}")
assert not _final_viols, "year-wide pre-check did not reach a feasible parameter set"

## Single-day smoke test — build price & demand for `MODEL_DAY`

The remaining sections still solve the **single-day** model as a smoke test. The day is
`MODEL_DAY` (2024-03-18). Its price vector comes from the timezone-correct price layer and its
demand comes from the synthetic order book (there is no hand-picked pickup dict anymore).

In [ ]:
_model_date = pd.to_datetime(MODEL_DAY).date()
assert is_business_day(_model_date), f"{MODEL_DAY} is not a business day"

# price vector (EUR/MWh) for the operating window of MODEL_DAY (timezone-correct)
price = prices_for_date(_model_date)
assert len(price) == N_T

# demand for MODEL_DAY straight from the order book -> pickup vector per slot
_day_orders = orderbook[orderbook["date"] == _model_date]
pickup_vec = np.zeros(N_T)
for _, r in _day_orders.iterrows():
    pickup_vec[int(r["slot"])] += r["quantity"]

times = [slot_to_hhmm(t) for t in range(N_T)]
print(f"Smoke-test day: {MODEL_DAY}")
print(f"  price EUR/MWh: min {price.min():.2f}, max {price.max():.2f}, mean {price.mean():.2f}")
print(f"  pickups      : {[(r['time'], int(r['quantity'])) for _, r in _day_orders.iterrows()]}")
print(f"  total demand : {int(pickup_vec.sum())} modules "
      f"(on slots {sorted({int(s) for s in np.nonzero(pickup_vec)[0]})})")

## Model — Sets

`ConcreteModel` with a time index and **session index sets**. Core idea: every potential session
gets a binary variable `start[s, t, (mode)]` indicating whether session *s* starts in slot *t*.
The load at a later slot follows linearly as **template parameter × start binary** (no var×var).

In [ ]:
m = pyo.ConcreteModel()
m.T = pyo.Set(initialize=range(N_T))
MODES = ["normal", "eco"]

# valid start slots (a session must fit inside the operating window)
mill_idx  = [(s, t, mode) for s in range(MAX_MILL_RUNS_PER_DAY) for mode in MODES
             for t in range(N_T - DUR[mode] + 1)]
press_idx = [(s, t) for s in range(MAX_PRESS_RUNS_PER_DAY) for t in range(N_T - PRESS_DUR + 1)]
m.MILL  = pyo.Set(initialize=mill_idx,  dimen=3)
m.PRESS = pyo.Set(initialize=press_idx, dimen=2)
print(f"Mill start vars: {len(mill_idx)} | Press start vars: {len(press_idx)}")

### Params (time series as `pyo.Param`)
Price and pickup quantities are known before the solve → registered as `pyo.Param`.

In [ ]:
m.price  = pyo.Param(m.T, initialize=dict(enumerate(price)))       # EUR/MWh
m.pickup = pyo.Param(m.T, initialize=dict(enumerate(pickup_vec)))  # modules

## Variables

- `mill_start[s,t,mode]`, `press_start[s,t]` — **binary** (session starts in slot *t*).
- `H[t]`, `M[t]` — housing / module stock at slot end, with capacity bounds.

There is **no grid variable**: without PV, grid draw equals the machine load, so the cost is a
direct linear expression in the start binaries (see the Objective section).

In [ ]:
m.mill_start  = pyo.Var(m.MILL,  within=pyo.Binary)
m.press_start = pyo.Var(m.PRESS, within=pyo.Binary)
m.H    = pyo.Var(m.T, bounds=(0, STORAGE_CAPACITY_HOUSINGS))
m.M    = pyo.Var(m.T, bounds=(0, STORAGE_CAPACITY_MODULES))

## Constraints

Helper expressions (linear sums over binaries) for load, occupancy, activity, and start slot.

In [ ]:
def mill_load(t):
    return pyo.quicksum(m.mill_start[s, t0, mode] * mill_tpl[mode][t - t0]
                        for (s, t0, mode) in mill_idx if t0 <= t < t0 + DUR[mode])
def press_load(t):
    return pyo.quicksum(m.press_start[s, t0] * press_tpl[t - t0]
                        for (s, t0) in press_idx if t0 <= t < t0 + PRESS_DUR)
def total_load(t):
    return mill_load(t) + press_load(t)

def mill_active(s):   return pyo.quicksum(m.mill_start[ss, t0, mo] for (ss, t0, mo) in mill_idx if ss == s)
def press_active(s):  return pyo.quicksum(m.press_start[ss, t0] for (ss, t0) in press_idx if ss == s)
def mill_startslot(s):  return pyo.quicksum(t0 * m.mill_start[ss, t0, mo] for (ss, t0, mo) in mill_idx if ss == s)
def press_startslot(s): return pyo.quicksum(t0 * m.press_start[ss, t0] for (ss, t0) in press_idx if ss == s)

**(C1) Each session starts at most once** — sessions are optional; unused ones stay inactive.

In [ ]:
m.mill_once  = pyo.Constraint(range(MAX_MILL_RUNS_PER_DAY),  rule=lambda m, s: mill_active(s)  <= 1)
m.press_once = pyo.Constraint(range(MAX_PRESS_RUNS_PER_DAY), rule=lambda m, s: press_active(s) <= 1)

**(C2) No overlap on the same machine** — at each slot at most one session per machine may
"occupy" it. Occupancy = sum of starts whose run interval contains the slot.

In [ ]:
def mill_overlap_rule(m, t):
    return pyo.quicksum(m.mill_start[s, t0, mode] for (s, t0, mode) in mill_idx
                        if t0 <= t < t0 + DUR[mode]) <= 1
m.mill_overlap = pyo.Constraint(m.T, rule=mill_overlap_rule)

def press_overlap_rule(m, t):
    return pyo.quicksum(m.press_start[s, t0] for (s, t0) in press_idx
                        if t0 <= t < t0 + PRESS_DUR) <= 1
m.press_overlap = pyo.Constraint(m.T, rule=press_overlap_rule)

**(C4) Housing balance** — production at mill **end** (+100 in the completion slot), press consumes
**at start** (−40). `H[t] ≥ 0` (bound) prevents consuming more than available; `H[t] ≤ capacity`
caps the stock. The `t=0` edge case is handled explicitly.

In [ ]:
def housing_rule(m, t):
    completes = pyo.quicksum(m.mill_start[s, t0, mode] for (s, t0, mode) in mill_idx
                             if t0 + DUR[mode] == t)
    consumes  = pyo.quicksum(m.press_start[s, t0] for (s, t0) in press_idx if t0 == t)
    prev = INITIAL_STOCK_HOUSINGS if t == 0 else m.H[t - 1]
    return m.H[t] == prev + HOUSINGS_PER_MILL_RUN * completes \
                          - HOUSINGS_CONSUMED_PER_PRESS_RUN * consumes
m.housing_bal = pyo.Constraint(m.T, rule=housing_rule)

**(C5) Module balance & pickup deadlines** — modules are created at press **end** (+40), pickups
subtract at their slot. `M[t] ≥ 0` enforces that stock **covers** the pickup (including modules
completing in the same slot); `M[t] ≤ capacity` is the storage limit.

In [ ]:
def module_rule(m, t):
    completes = pyo.quicksum(m.press_start[s, t0] for (s, t0) in press_idx if t0 + PRESS_DUR == t)
    prev = INITIAL_STOCK_MODULES if t == 0 else m.M[t - 1]
    return m.M[t] == prev + MODULES_PER_PRESS_RUN * completes - m.pickup[t]
m.module_bal = pyo.Constraint(m.T, rule=module_rule)

**(C6) Symmetry breaking** — the interchangeable session indices would blow up the search space.
We enforce: session *s* active only if *s−1* is active, and **increasing start times** (big-M,
active only when the session is used).

In [ ]:
m.mill_sym  = pyo.Constraint(range(1, MAX_MILL_RUNS_PER_DAY),  rule=lambda m, s: mill_active(s)  <= mill_active(s - 1))
m.press_sym = pyo.Constraint(range(1, MAX_PRESS_RUNS_PER_DAY), rule=lambda m, s: press_active(s) <= press_active(s - 1))
m.mill_ord  = pyo.Constraint(range(1, MAX_MILL_RUNS_PER_DAY),
    rule=lambda m, s: mill_startslot(s)  >= mill_startslot(s - 1)  + 1 - N_T * (1 - mill_active(s)))
m.press_ord = pyo.Constraint(range(1, MAX_PRESS_RUNS_PER_DAY),
    rule=lambda m, s: press_startslot(s) >= press_startslot(s - 1) + 1 - N_T * (1 - press_active(s)))

## Objective

Minimize **electricity cost** = machine load [kW] × 0.25 h × price [EUR/MWh] / 1000 (→ EUR),
summed over all slots:

$$\min \sum_{t} \text{load}[t]\cdot \text{STEP\_H}\cdot \frac{\text{price}[t]}{1000}$$

Without PV there is no `max(0, load − PV)` term to linearise, so the cost is a direct linear
expression in the start binaries. **Negative prices** (5.2 % of 2024 slots) are handled correctly:
they lower the cost of producing in that slot and simply become attractive production windows.
Because production is bounded by the demand, storage and once-per-session constraints, the problem
stays **bounded** even at negative prices — unlike the old PV formulation.

In [ ]:
m.obj = pyo.Objective(
    expr=pyo.quicksum(total_load(t) * STEP_H * m.price[t] / 1000.0 for t in m.T),
    sense=pyo.minimize)

## Run Solver

The structural feasibility was already established year-wide by the auto-tuned pre-check above, so
here we simply solve the single-day smoke-test model with HiGHS and assert an optimal status.

In [ ]:
solver = pyo.SolverFactory("appsi_highs")
result = solver.solve(m, tee=False)
status = result.solver.status; term = result.solver.termination_condition
print("Solver:", status, "|", term)
if term == pyo.TerminationCondition.infeasible:
    print("Model infeasible for the smoke-test day.")
else:
    assert status == pyo.SolverStatus.ok
    assert term == pyo.TerminationCondition.optimal
    print(f"Optimal solution — electricity cost: {pyo.value(m.obj):.4f} EUR")

## Extract Results

Per-slot DataFrame: machine load, price, cost, and inventory levels.

In [ ]:
def val(v): return float(pyo.value(v))
mill_load_v  = np.array([val(mill_load(t))  for t in range(N_T)])
press_load_v = np.array([val(press_load(t)) for t in range(N_T)])
H_v          = np.array([val(m.H[t])        for t in range(N_T)])
M_v          = np.array([val(m.M[t])        for t in range(N_T)])
load_v       = mill_load_v + press_load_v
cost_v       = load_v * STEP_H * price / 1000.0            # grid draw = load (no PV)

results_df = pd.DataFrame({
    "time": times, "mill_kW": mill_load_v, "press_kW": press_load_v, "total_load_kW": load_v,
    "price_eur_mwh": price, "cost_eur": cost_v,
    "housings": H_v, "modules": M_v, "pickup": pickup_vec,
}).round(4)   # written to the run folder in the save-outputs block (after the baseline)

# active sessions
mill_runs = sorted([(t, mode) for (s, t, mode) in mill_idx if val(m.mill_start[s, t, mode]) > 0.5])
press_runs = sorted([t for (s, t) in press_idx if val(m.press_start[s, t]) > 0.5])
print("Mill runs :", [(slot_to_hhmm(t), mode) for t, mode in mill_runs])
print("Press runs:", [slot_to_hhmm(t) for t in press_runs])
print("Total cost:", round(cost_v.sum(), 4), "EUR | grid draw",
      round(load_v.sum()*STEP_H, 2), "kWh")
results_df.head()

### Baseline for the cost comparison

With PV removed, only **two scenarios** remain:

1. **optimized** (`cost_opt`) — the MILP result above.
2. **asap** (`cost_asap`) — the price-blind baseline: every run placed as early as technically
   possible (normal mode only), respecting the same pickup deadlines. This isolates the value of
   load flexibility (shifting out of expensive hours) against a naive schedule.

In [ ]:
# Naive ASAP schedule (normal mode, price-blind)
def asap_schedule():
    total_pick = int(pickup_vec.sum())
    n_press = int(np.ceil((total_pick - INITIAL_STOCK_MODULES) / MODULES_PER_PRESS_RUN))
    n_press = max(0, n_press)
    n_mill  = int(np.ceil((n_press * HOUSINGS_CONSUMED_PER_PRESS_RUN - INITIAL_STOCK_HOUSINGS)
                          / HOUSINGS_PER_MILL_RUN))
    n_mill = max(0, n_mill)
    mill_l = np.zeros(N_T); press_l = np.zeros(N_T)
    mill_iv = []; press_iv = []                 # run intervals (slot, duration[, mode])
    H = INITIAL_STOCK_HOUSINGS
    comp = {}                                   # slot -> housing increment
    # mills back-to-back from slot 0 (normal mode)
    t = 0
    for _ in range(n_mill):
        if t + DUR["normal"] > N_T: break
        for k in range(DUR["normal"]): mill_l[t + k] += mill_normal[k]
        mill_iv.append((t, DUR["normal"], "normal"))
        comp[t + DUR["normal"]] = comp.get(t + DUR["normal"], 0) + HOUSINGS_PER_MILL_RUN
        t += DUR["normal"]
    # presses ASAP: machine free + enough housings
    placed = 0; busy_until = 0
    for t in range(N_T):
        H += comp.get(t, 0)
        if placed < n_press and t >= busy_until and H >= HOUSINGS_CONSUMED_PER_PRESS_RUN \
           and t + PRESS_DUR <= N_T:
            for k in range(PRESS_DUR): press_l[t + k] += press_tpl[k]
            press_iv.append((t, PRESS_DUR))
            H -= HOUSINGS_CONSUMED_PER_PRESS_RUN; busy_until = t + PRESS_DUR; placed += 1
    return mill_l + press_l, placed, n_press, mill_iv, press_iv

asap_load, placed, n_press_need, base_mill_iv, base_press_iv = asap_schedule()
cost_asap = float((asap_load * STEP_H * price / 1000.0).sum())
cost_opt  = float(cost_v.sum())

print(f"Cost optimized        : {cost_opt:7.4f} EUR")
print(f"Cost ASAP (baseline)  : {cost_asap:7.4f} EUR  (flexibility saving {cost_asap-cost_opt:+.4f})")
if placed < n_press_need:
    print(f"  ! ASAP could only place {placed}/{n_press_need} press runs.")

### Save run outputs (per-run folder)

Each run gets its own folder under `results/` (`RUN_DIR`, created in the Parameters section). This
block writes the two schedule CSVs (**optimized**, **asap**) and a **`parameters.json`** that
records everything needed to reproduce the run: all tunable (post-tuning) and fixed parameters, the
seed, the order-book file and its summary statistics, the price file, the study year, the derived
slot durations, and the template file names.

In [ ]:
def build_schedule_df(mill_iv, press_iv):
    "Per-slot schedule DataFrame from run intervals (same booking logic as the model, no PV)."
    ml = np.zeros(N_T); pl = np.zeros(N_T)
    mill_comp = {}; press_comp = {}; press_st = {}
    for iv in mill_iv:
        t, d, mode = iv[0], iv[1], iv[2]
        for k in range(d):
            if t + k < N_T: ml[t + k] += mill_tpl[mode][k]
        mill_comp[t + d] = mill_comp.get(t + d, 0) + 1
    for iv in press_iv:
        t = iv[0]
        for k in range(PRESS_DUR):
            if t + k < N_T: pl[t + k] += press_tpl[k]
        press_st[t] = press_st.get(t, 0) + 1
        press_comp[t + PRESS_DUR] = press_comp.get(t + PRESS_DUR, 0) + 1
    H = np.zeros(N_T); M = np.zeros(N_T); h = INITIAL_STOCK_HOUSINGS; mm = INITIAL_STOCK_MODULES
    for t in range(N_T):
        h  += HOUSINGS_PER_MILL_RUN * mill_comp.get(t, 0) \
              - HOUSINGS_CONSUMED_PER_PRESS_RUN * press_st.get(t, 0)
        mm += MODULES_PER_PRESS_RUN * press_comp.get(t, 0) - pickup_vec[t]
        H[t] = h; M[t] = mm
    load = ml + pl
    return pd.DataFrame({"time": times, "mill_kW": ml, "press_kW": pl, "total_load_kW": load,
        "price_eur_mwh": price, "cost_eur": load * STEP_H * price / 1000.0,
        "housings": H, "modules": M, "pickup": pickup_vec}).round(4)

# 1) optimized (model-exact)
results_df.to_csv(os.path.join(RUN_DIR, "schedule_optimized.csv"), index=False)
# 2) asap baseline
df_asap = build_schedule_df(base_mill_iv, base_press_iv)
df_asap.to_csv(os.path.join(RUN_DIR, "schedule_asap.csv"), index=False)

# consistency check against the cost computed in the baseline block
# (tolerance > rounding: CSV values are rounded to 4 decimals)
assert abs(df_asap["cost_eur"].sum() - cost_asap) < 1e-2

# --- Parameter log: full reproducible configuration as JSON ----------------
run_params = {
    "study_year": STUDY_YEAR,
    "seed": SEED,
    "model_day": MODEL_DAY,
    "labels": {"util_note": UTIL_NOTE},
    "time": {"workday_start": WORKDAY_START, "workday_end": WORKDAY_END,
             "step_min": STEP_MIN, "n_slots": N_T},
    "production": {
        "housings_per_mill_run": HOUSINGS_PER_MILL_RUN,
        "modules_per_press_run": MODULES_PER_PRESS_RUN,
        "housings_consumed_per_press_run": HOUSINGS_CONSUMED_PER_PRESS_RUN,
    },
    "inventory": {
        "initial_stock_housings": INITIAL_STOCK_HOUSINGS,
        "initial_stock_modules": INITIAL_STOCK_MODULES,
        "storage_capacity_housings": STORAGE_CAPACITY_HOUSINGS,
        "storage_capacity_modules": STORAGE_CAPACITY_MODULES,
    },
    "eco_mode": {
        "power_scaling": ECO_MODE_POWER_SCALING,
        "time_stretch": ECO_MODE_TIME_STRETCH,
    },
    "session_upper_bounds": {
        "max_mill_runs_per_day": MAX_MILL_RUNS_PER_DAY,
        "max_press_runs_per_day": MAX_PRESS_RUNS_PER_DAY,
        "max_mill_runs_per_window": MAX_MILL_RUNS_PER_WINDOW,
        "max_press_runs_per_window": MAX_PRESS_RUNS_PER_WINDOW,
        "mill_runs_ceiling": MILL_RUNS_CEILING,
        "press_runs_ceiling": PRESS_RUNS_CEILING,
    },
    "derived_slot_durations": {
        "mill_normal": DUR["normal"], "mill_eco": DUR["eco"], "press": PRESS_DUR,
    },
    "auto_tuning": {
        "adjustments": tuning_log,
        "n_adjustments": len(tuning_log),
    },
    "order_book": {"file": OB_FILE, "meta_file": OB_META_FILE, "generator": OB_DIST, "stats": OB_STATS},
    "price_file": PRICE_FILE,
    "calendar_file": "data/business_days.csv",
    "template_files": {
        "mill":  "chiron_session6_min230-380_1min.csv",
        "press": "chippress_session4_window_1min.csv",
    },
}
with open(os.path.join(RUN_DIR, "parameters.json"), "w", encoding="utf-8") as f:
    json.dump(run_params, f, indent=2, ensure_ascii=False)

print("Run folder:", RUN_DIR)
print("Written: schedule_optimized.csv, schedule_asap.csv, parameters.json")
print("(Plots are written to the same folder by the 'Plot Results' section.)")

## Plot Results

### Schedule: machine load and price

In [ ]:
x = list(range(N_T))
# pickup deadlines (slot -> quantity) for this day, from the order book
pickup_marks = {int(s): float(pickup_vec[s]) for s in np.nonzero(pickup_vec)[0]}

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=x, y=mill_load_v, name="Mill", marker_color=C["mill"], opacity=0.85), secondary_y=False)
fig.add_trace(go.Bar(x=x, y=press_load_v, name="Press", marker_color=C["press"], opacity=0.85), secondary_y=False)
fig.add_trace(go.Scatter(x=x, y=price, name="Price", mode="lines",
                         line=dict(color=C["price"], width=2, dash="dot")), secondary_y=True)
for s in pickup_marks:
    fig.add_vline(x=s, line=dict(color="black", dash="dash", width=1))
fig.update_layout(barmode="stack", template="plotly_white", height=460, width=1050,
                  title=f"Optimized schedule {MODEL_DAY} — load (stacked) & day-ahead price",
                  xaxis=dict(tickmode="array", tickvals=x[::4], ticktext=times[::4], title="Time"))
fig.update_yaxes(title_text="Power [kW]", secondary_y=False)
fig.update_yaxes(title_text="Price [EUR/MWh]", secondary_y=True, showgrid=False)
fig.write_image(f"{RUN_DIR}/opt_schedule.png", scale=2)
fig.show()

### Inventory levels over the day (with pickup deadlines)

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=x, y=H_v, name="Housings (intermediate)", mode="lines+markers",
                         line=dict(color=C["housing"], width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=x, y=M_v, name="Modules (finished)", mode="lines+markers",
                         line=dict(color=C["module"], width=2)), secondary_y=True)
fig.add_hline(y=STORAGE_CAPACITY_HOUSINGS, line=dict(color=C["housing"], dash="dot", width=1),
              annotation_text="Housing cap.", annotation_position="top left", secondary_y=False)
for s, q in pickup_marks.items():
    fig.add_vline(x=s, line=dict(color="black", dash="dash", width=1))
    fig.add_annotation(x=s, y=q, yref="y2", text=f"Pickup {int(q)}",
                       showarrow=True, arrowhead=2, ax=-28, ay=38)
fig.update_layout(template="plotly_white", height=440, width=1050,
                  title="Inventory levels — housings & modules (pickup deadlines dashed)",
                  xaxis=dict(tickmode="array", tickvals=x[::4], ticktext=times[::4], title="Time"))
fig.update_yaxes(title_text="Housings [pcs]", secondary_y=False)
fig.update_yaxes(title_text="Modules [pcs]", secondary_y=True, showgrid=False)
fig.write_image(f"{RUN_DIR}/opt_inventory.png", scale=2)
fig.show()

### Cost comparison — optimized vs. ASAP baseline

In [ ]:
# Two scenarios: optimized (blue) vs. price-blind ASAP baseline (orange).
labels = ["Optimized", "ASAP (baseline)"]
values = [cost_opt, cost_asap]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=labels, y=values, marker_color=[C["mill"], C["press"]],
    text=[f"{v:.2f} EUR" for v in values], textposition="outside"))
fig.update_layout(
    template="plotly_white", height=460, width=620,
    title=f"Electricity cost {MODEL_DAY} — optimized vs. ASAP",
    yaxis_title="Cost [EUR]", showlegend=False)
fig.update_yaxes(range=[min(0, min(values) * 1.1), max(values) * 1.18])
fig.write_image(f"{RUN_DIR}/opt_cost_breakdown.png", scale=2)
fig.show()

### Dispatch overview: baseline vs. optimized

A one-dimensional timeline of the 14-hour workday. Colored bars show **when each machine runs** —
top: the unoptimized **baseline (ASAP)** scenario, bottom: the **optimized** scenario. This makes
the load-shifting decision visible at a glance (mill = blue, press = orange; pickup deadlines
dashed).

In [ ]:
# Run intervals per scenario: (start slot, duration slots, mode)
opt_mill_iv  = [(t, DUR[mode], mode) for (t, mode) in mill_runs]
opt_press_iv = [(t, PRESS_DUR, None) for t in press_runs]
b_mill_iv    = list(base_mill_iv)                       # (t, dur, "normal")
b_press_iv   = [(t, d, None) for (t, d) in base_press_iv]

lanes = [   # (lane label, intervals, machine key)
    ("Baseline · Mill",    b_mill_iv,    "mill"),
    ("Baseline · Press",   b_press_iv,   "press"),
    ("Optimized · Mill",   opt_mill_iv,  "mill"),
    ("Optimized · Press",  opt_press_iv, "press"),
]
# category order bottom->top: optimized at the bottom, baseline on top
order = ["Optimized · Press", "Optimized · Mill", "Baseline · Press", "Baseline · Mill"]

fig = go.Figure()
seen = set()
for label, ivs, mk in lanes:
    if not ivs:
        continue
    starts = [t for (t, d, _) in ivs]
    durs   = [d for (t, d, _) in ivs]
    txt    = [("Eco" if lab == "eco" else "N") if mk == "mill" else "" for (_, _, lab) in ivs]
    fig.add_trace(go.Bar(
        y=[label]*len(ivs), x=durs, base=starts, orientation="h", width=0.62,
        marker=dict(color=C[mk], line=dict(color="white", width=1.2)),
        text=txt, textposition="inside", insidetextanchor="middle",
        textfont=dict(color="white", size=10),
        legendgroup=mk, name=("Mill" if mk == "mill" else "Press"), showlegend=(mk not in seen),
        hovertemplate="%{y}<br>start %{customdata}, duration %{x} slots<extra></extra>",
        customdata=[slot_to_hhmm(t) for t in starts]))
    seen.add(mk)

for s in pickup_marks:
    fig.add_vline(x=s, line=dict(color="black", dash="dash", width=1))
    fig.add_annotation(x=s, y=1.04, yref="paper", showarrow=False,
                       text=f"Pickup {slot_to_hhmm(s)}", font=dict(size=10, color="black"))
# separator between the two scenarios
fig.add_hline(y=1.5, line=dict(color="#bbbbbb", width=1))
fig.update_yaxes(categoryorder="array", categoryarray=order)
fig.update_layout(barmode="overlay", template="plotly_white", height=340, width=1050,
    title="Machine dispatch: baseline (ASAP, top) vs. optimized (bottom)",
    bargap=0.35, legend=dict(orientation="h", y=-0.22),
    xaxis=dict(range=[0, N_T], tickmode="array", tickvals=list(range(0, N_T+1, 4)),
               ticktext=[slot_to_hhmm(t) for t in range(0, N_T+1, 4)], title="Time"))
fig.write_image(f"{RUN_DIR}/opt_dispatch_comparison.png", scale=2)
fig.show()

## Summary

The following cell summarizes the decisions in plain language (derived dynamically from the solution).

In [ ]:
n_eco = sum(1 for _, mode in mill_runs if mode == "eco")
flex_save = cost_asap - cost_opt
print("="*66)
print(f"RESULT — smoke-test day {MODEL_DAY} (08:00-22:00 local, 15-min)")
print("="*66)
print(f"Electricity cost (optimized): {cost_opt:.2f} EUR")
print(f"Electricity cost (ASAP)     : {cost_asap:.2f} EUR")
print(f"  - flexibility saving      : {flex_save:.2f} EUR "
      f"({flex_save/cost_asap*100:.1f} % vs. ASAP)" if cost_asap else "")
print(f"Mill runs : {len(mill_runs)}  (of which {n_eco} in eco mode)")
print(f"Press runs: {len(press_runs)}")
print(f"Peak load : {load_v.max():.1f} kW | grid draw {load_v.sum()*STEP_H:.1f} kWh")
print(f"Mill start times : {[slot_to_hhmm(t) for t,_ in mill_runs]}")
print(f"Press start times: {[slot_to_hhmm(t) for t in press_runs]}")
print("All pickups met:", bool((M_v >= -1e-6).all()))

**Reading.** The solver places the runs in **cheap 15-min slots** (including any negative-price
windows) while respecting per-machine non-overlap, storage capacities, and the pickup deadlines,
and selects **eco mode** for individual mill runs when the lower peak load (×0.75) outweighs the
longer occupancy (×1.25) in cost terms. The single remaining comparison is **optimized vs. the
price-blind ASAP baseline**, which isolates the value of load flexibility — shifting production out
of expensive hours.

*Note:* this is only the **single-day smoke test** for `MODEL_DAY`. The full-year rolling-horizon
run, the business case, and the paper figures are the subject of the next study steps.

# Rolling two-day horizon — full-year 2024 run

Step 2 of 3. The single-day smoke test above stays as a sanity check; this section runs the actual
study: a **rolling two-production-day window** over all 252 business days of 2024.

**Window.** Day `D` is the business day whose schedule is committed; day `D' = next_business_day(D)`
is the look-ahead day (a Friday looks ahead to the following Monday). Slots `0..55` are day `D`
(08:00–21:45 local), slots `56..111` are day `D'`. There are **no slots between 22:00 on D and
08:00 on D'** — the window is not contiguous in wall-clock time; inventory simply carries across the
gap. Price and pickup vectors are the concatenation of the two 56-slot local-time slices.

**Day-boundary rule.** A session must not span the boundary: valid mill/press starts live entirely
inside one day segment (`[0, 56-DUR]` ∪ `[56, 112-DUR]`), never the naive `[0, 112-DUR]` — that
would let a run started at 21:00 keep producing overnight.

**Commit & carry.** Only day `D` is committed. Because production is booked at session **end**, the
last runs of day `D` (a mill starting at slot 46 occupies 46..55 and credits at slot **56**, the
first slot of `D'`) must be added to the carry-over stock explicitly — reading `H[55]`/`M[55]`
alone would silently discard every day's end-of-shift batch. The carried stock seeds the next
window's `init_H`/`init_M`.

Both the **optimized** MILP and a **price-blind ASAP baseline** are rolled through the year with
their own inventory chains, starting from the same initial stock, satisfying exactly the same
pickups. Nothing here is monetised beyond electricity cost — the business case is step 3.

In [ ]:
# --- Rolling-horizon window model (self-contained; does not touch the single-day `m`) ---
from collections import defaultdict
from time import perf_counter

SLOTS_PER_DAY = N_T                        # 56 slots per production day
# per-run energies (kWh) for the energy-conservation check
E_MILL_NORMAL = float(mill_normal.sum() * STEP_H)
E_MILL_ECO    = float(mill_eco.sum()    * STEP_H)
E_PRESS       = float(press_tpl.sum()   * STEP_H)

def _window_valid_starts(dur, n_days):
    "Start slots that keep a run of length `dur` inside a single day segment (day-boundary rule)."
    starts = []
    for k in range(n_days):
        seg0 = k * SLOTS_PER_DAY
        starts.extend(range(seg0, seg0 + SLOTS_PER_DAY - dur + 1))
    return starts

def _build_window_vectors(date_D, date_Dprime):
    "Concatenate the two days' 56-slot local-time price and pickup slices (D' optional)."
    days = [date_D] + ([date_Dprime] if date_Dprime is not None else [])
    price = np.concatenate([prices_for_date(d) for d in days])
    pickup = np.zeros(len(days) * SLOTS_PER_DAY)
    for k, d in enumerate(days):
        for _, r in orderbook[orderbook["date"] == d].iterrows():
            pickup[k * SLOTS_PER_DAY + int(r["slot"])] += r["quantity"]
    return price, pickup, len(days)

def solve_window(date_D, date_Dprime, init_H, init_M, time_limit=60):
    "Build and solve the window MILP. Returns a result dict; `feasible` is False if not optimal."
    price, pickup, n_days = _build_window_vectors(date_D, date_Dprime)
    N = n_days * SLOTS_PER_DAY
    mill_cap  = MAX_MILL_RUNS_PER_DAY  * n_days     # window-level caps (8 / 20 for a 2-day window)
    press_cap = MAX_PRESS_RUNS_PER_DAY * n_days

    mill_index  = [(s, t, mode) for s in range(mill_cap) for mode in MODES
                   for t in _window_valid_starts(DUR[mode], n_days)]
    press_index = [(s, t) for s in range(press_cap)
                   for t in _window_valid_starts(PRESS_DUR, n_days)]

    # occupancy / event maps (built once -> model construction stays linear in total occupancy)
    mill_occ = defaultdict(list); mill_done = defaultdict(list); mill_by_s = defaultdict(list)
    for (s, t, mode) in mill_index:
        for k in range(DUR[mode]):
            mill_occ[t + k].append((s, t, mode, mill_tpl[mode][k]))
        mill_done[t + DUR[mode]].append((s, t, mode))
        mill_by_s[s].append((t, mode))
    press_occ = defaultdict(list); press_done = defaultdict(list)
    press_start_at = defaultdict(list); press_by_s = defaultdict(list)
    for (s, t) in press_index:
        for k in range(PRESS_DUR):
            press_occ[t + k].append((s, t, press_tpl[k]))
        press_done[t + PRESS_DUR].append((s, t))
        press_start_at[t].append((s, t))
        press_by_s[s].append(t)

    mw = pyo.ConcreteModel()
    mw.T     = pyo.Set(initialize=range(N))
    mw.MILL  = pyo.Set(initialize=mill_index,  dimen=3)
    mw.PRESS = pyo.Set(initialize=press_index, dimen=2)
    mw.mill_start  = pyo.Var(mw.MILL,  within=pyo.Binary)
    mw.press_start = pyo.Var(mw.PRESS, within=pyo.Binary)
    mw.H = pyo.Var(mw.T, bounds=(0, STORAGE_CAPACITY_HOUSINGS))
    mw.M = pyo.Var(mw.T, bounds=(0, STORAGE_CAPACITY_MODULES))

    def tload(t):
        return (pyo.quicksum(c * mw.mill_start[s, t0, mode] for (s, t0, mode, c) in mill_occ.get(t, []))
                + pyo.quicksum(c * mw.press_start[s, t0] for (s, t0, c) in press_occ.get(t, [])))

    # (C1) at most one start per session
    mw.mill_once  = pyo.Constraint(range(mill_cap),
        rule=lambda mw, s: pyo.quicksum(mw.mill_start[s, t, mode] for (t, mode) in mill_by_s[s]) <= 1)
    mw.press_once = pyo.Constraint(range(press_cap),
        rule=lambda mw, s: pyo.quicksum(mw.press_start[s, t] for t in press_by_s[s]) <= 1)
    # (C2) no overlap per machine
    mw.mill_overlap = pyo.Constraint(mw.T,
        rule=lambda mw, t: pyo.quicksum(mw.mill_start[s, t0, mode] for (s, t0, mode, c) in mill_occ.get(t, [])) <= 1)
    mw.press_overlap = pyo.Constraint(mw.T,
        rule=lambda mw, t: pyo.quicksum(mw.press_start[s, t0] for (s, t0, c) in press_occ.get(t, [])) <= 1)
    # (C4) housing balance (production at mill end, consumption at press start)
    def housing_rule(mw, t):
        completes = pyo.quicksum(mw.mill_start[s, t0, mode] for (s, t0, mode) in mill_done.get(t, []))
        consumes  = pyo.quicksum(mw.press_start[s, t0] for (s, t0) in press_start_at.get(t, []))
        prev = init_H if t == 0 else mw.H[t - 1]
        return mw.H[t] == prev + HOUSINGS_PER_MILL_RUN * completes - HOUSINGS_CONSUMED_PER_PRESS_RUN * consumes
    mw.housing_bal = pyo.Constraint(mw.T, rule=housing_rule)
    # (C5) module balance & pickups (production at press end)
    def module_rule(mw, t):
        completes = pyo.quicksum(mw.press_start[s, t0] for (s, t0) in press_done.get(t, []))
        prev = init_M if t == 0 else mw.M[t - 1]
        return mw.M[t] == prev + MODULES_PER_PRESS_RUN * completes - float(pickup[t])
    mw.module_bal = pyo.Constraint(mw.T, rule=module_rule)
    # (C6) symmetry breaking. Only the cheap "active(s) <= active(s-1)" ordering is used here:
    # it breaks the session-index permutation symmetry at negligible cost. The big-M start-slot
    # ordering from the single-day model is deliberately OMITTED — over 252 window solves it is a
    # major slowdown, and it is pure symmetry breaking, so it changes neither the optimal cost nor
    # the committed schedule.
    mw.mill_sym = pyo.Constraint(range(1, mill_cap),
        rule=lambda mw, s: pyo.quicksum(mw.mill_start[s, t, mo] for (t, mo) in mill_by_s[s])
                           <= pyo.quicksum(mw.mill_start[s - 1, t, mo] for (t, mo) in mill_by_s[s - 1]))
    mw.press_sym = pyo.Constraint(range(1, press_cap),
        rule=lambda mw, s: pyo.quicksum(mw.press_start[s, t] for t in press_by_s[s])
                           <= pyo.quicksum(mw.press_start[s - 1, t] for t in press_by_s[s - 1]))

    mw.obj = pyo.Objective(expr=pyo.quicksum(tload(t) * STEP_H * price[t] / 1000.0 for t in range(N)),
                           sense=pyo.minimize)

    solver = pyo.SolverFactory("appsi_highs")
    try:
        solver.config.time_limit = time_limit
        solver.config.mip_gap = 0.0
    except Exception:
        pass
    # load_solutions=False: appsi_highs RAISES on an infeasible model if it tries to load a
    # (non-existent) solution. Defer loading so we can inspect the termination condition and let
    # the year loop emit its own diagnostic instead of a raw solver exception.
    t_start = perf_counter()
    res = solver.solve(mw, load_solutions=False)
    solve_s = perf_counter() - t_start
    term = res.solver.termination_condition

    out = {"date_D": date_D, "date_Dprime": date_Dprime, "n_days": n_days, "N": N,
           "price": price, "pickup": pickup, "termination": str(term), "solve_s": solve_s,
           "hit_time_limit": (term == pyo.TerminationCondition.maxTimeLimit),
           "feasible": (term == pyo.TerminationCondition.optimal)}
    if not out["feasible"]:
        return out
    solver.load_vars()   # only safe once we know a solution exists

    v = lambda var: float(pyo.value(var))
    mill_runs  = sorted([(t, mode) for (s, t, mode) in mill_index if v(mw.mill_start[s, t, mode]) > 0.5])
    press_runs = sorted([t for (s, t) in press_index if v(mw.press_start[s, t]) > 0.5])
    mill_load_v = np.zeros(N); press_load_v = np.zeros(N)
    for (t0, mode) in mill_runs:
        for k in range(DUR[mode]): mill_load_v[t0 + k] += mill_tpl[mode][k]
    for t0 in press_runs:
        for k in range(PRESS_DUR): press_load_v[t0 + k] += press_tpl[k]
    # day-boundary safety net: no active run crosses index 56
    for (t0, mode) in mill_runs:
        assert (t0 // SLOTS_PER_DAY) == ((t0 + DUR[mode] - 1) // SLOTS_PER_DAY), "mill run crosses day boundary"
    for t0 in press_runs:
        assert (t0 // SLOTS_PER_DAY) == ((t0 + PRESS_DUR - 1) // SLOTS_PER_DAY), "press run crosses day boundary"
    out.update({"mill_load": mill_load_v, "press_load": press_load_v,
                "H": np.array([v(mw.H[t]) for t in range(N)]),
                "M": np.array([v(mw.M[t]) for t in range(N)]),
                "mill_runs": mill_runs, "press_runs": press_runs, "obj": float(pyo.value(mw.obj))})
    return out

In [ ]:
# --- Price-blind ASAP baseline in the SAME rolling framework -----------------
# INDEPENDENCE RULE. The baseline is completely independent of the optimisation. It
# never reads `price[t]` in any form and never reads any MILP result, variable value,
# run count or schedule -- reusing the optimum's quantities would hand the baseline
# the very foresight this study is trying to value. Its only inputs are the ORDER
# BOOK and its OWN inventory state.
#
# Look-ahead over DEMAND is allowed and in fact necessary: a 190-module pickup at
# 15:00 needs five press runs finished by slot 28, i.e. 200 housings, while the opening
# stock is 60 and a mill run only credits its 100 housings at slot 10. So the baseline
# gets the same two-day demand visibility the MILP has -- visibility of DEMAND, not of
# decisions or prices.

PRESS_STARTS_MAX = SLOTS_PER_DAY // PRESS_DUR    # 11 press runs fit into one day
MILL_STARTS_ASAP = [k * DUR["normal"] for k in range(SLOTS_PER_DAY // DUR["normal"])]  # 0,10,20,30,40


def _window_pickups(date_D, date_Dprime):
    "Per-day 56-slot pickup vectors of the window. Demand only -- no price is touched."
    days = [date_D] + ([date_Dprime] if date_Dprime is not None else [])
    segs = []
    for d in days:
        seg = np.zeros(SLOTS_PER_DAY)
        for _, r in orderbook[orderbook["date"] == d].iterrows():
            seg[int(r["slot"])] += r["quantity"]
        segs.append(seg)
    return segs


def _asap_day(pickup_seg, H0, M0, n_press):
    """Place `n_press` press runs -- and the MINIMUM number of normal-mode mill runs that
    feeds them -- as early as possible inside ONE 56-slot production day.

    Mill runs go back-to-back from slot 0 (earliest possible, never eco). A press run is
    delayed past its earliest slot only when the press is busy, when no housings are on
    stock, or when crediting its 40 modules would breach STORAGE_CAPACITY_MODULES -- ASAP
    piles up finished goods by construction, so without the cap check it would breach it.
    Returns None if the runs do not fit into the day (no run may cross the day boundary).

    Booking convention is the model's: production credits at session END, press
    consumption at session START. Index 56 is the carry bucket -- a batch finishing at the
    end of the shift is available the next morning, not on the same day.
    """
    n_mill = max(0, int(np.ceil(
        (n_press * HOUSINGS_CONSUMED_PER_PRESS_RUN - H0) / HOUSINGS_PER_MILL_RUN)))
    if n_mill > len(MILL_STARTS_ASAP):
        return None
    mill_runs = [(t0, "normal") for t0 in MILL_STARTS_ASAP[:n_mill]]
    h_credit = defaultdict(int)
    for (t0, mode) in mill_runs:
        h_credit[t0 + DUR[mode]] += HOUSINGS_PER_MILL_RUN

    # module trajectory, index 56 = carry bucket; press credits are added while placing
    Mpath = np.empty(SLOTS_PER_DAY + 1)
    mm = float(M0)
    for t in range(SLOTS_PER_DAY):
        mm -= float(pickup_seg[t])
        Mpath[t] = mm
    Mpath[SLOTS_PER_DAY] = mm

    press_runs = []
    h = float(H0)
    busy = 0
    for t in range(SLOTS_PER_DAY):
        h += h_credit.get(t, 0)
        if (len(press_runs) < n_press and t >= busy
                and h >= HOUSINGS_CONSUMED_PER_PRESS_RUN
                and t + PRESS_DUR <= SLOTS_PER_DAY):
            Mpath[t + PRESS_DUR:] += MODULES_PER_PRESS_RUN
            if Mpath.max() > STORAGE_CAPACITY_MODULES + 1e-9:
                Mpath[t + PRESS_DUR:] -= MODULES_PER_PRESS_RUN   # store full -> delay this run
            else:
                press_runs.append(t)
                h -= HOUSINGS_CONSUMED_PER_PRESS_RUN
                busy = t + PRESS_DUR
    if len(press_runs) < n_press:
        return None

    press_at = defaultdict(int)
    for t0 in press_runs:
        press_at[t0] += 1
    Hpath = np.empty(SLOTS_PER_DAY)
    hh = float(H0)
    for t in range(SLOTS_PER_DAY):
        hh += h_credit.get(t, 0) - HOUSINGS_CONSUMED_PER_PRESS_RUN * press_at.get(t, 0)
        Hpath[t] = hh
    return {"mill_runs": mill_runs, "press_runs": press_runs,
            "H": Hpath, "M": Mpath[:SLOTS_PER_DAY],
            "carry_H": hh + h_credit.get(SLOTS_PER_DAY, 0),
            "carry_M": float(Mpath[SLOTS_PER_DAY]),
            "served": bool(Mpath[:SLOTS_PER_DAY].min() >= -1e-9)}


def _asap_required_press(pickup_D, pickup_Dp, H0, M0):
    """MINIMUM number of press runs on day D -- not one run more -- such that

      (a) every day-D pickup is served from M0 plus what completes in time, and
      (b) day D' can still serve its own pickups out of the resulting carry stock while
          running its own machines as early as possible.

    (b) is the only reason the baseline pre-produces at all, and it pre-produces only the
    part of D' that D' provably cannot make in time itself -- never the whole of D'.

    The requirement is recomputed FROM SCRATCH every day from the ACTUAL current stock.
    That is the anti-ratchet property: anything produced ahead lowers today's requirement
    by exactly as much, so a surplus cannot accumulate over the year.
    """
    for n in range(PRESS_STARTS_MAX + 1):
        simD = _asap_day(pickup_D, H0, M0, n)
        if simD is None or not simD["served"]:
            continue
        if pickup_Dp is None:
            return n, simD, None
        need = int(np.ceil(max(0.0, float(pickup_Dp.sum()) - simD["carry_M"])
                           / MODULES_PER_PRESS_RUN))
        for k in range(need, PRESS_STARTS_MAX + 1):
            simP = _asap_day(pickup_Dp, simD["carry_H"], simD["carry_M"], k)
            if simP is not None and simP["served"]:
                return n, simD, simP
    return None, None, None


def asap_window(date_D, date_Dprime, init_H, init_M):
    "ASAP heuristic over the window (normal mode only, never looks at price). Same return shape as solve_window."
    segs = _window_pickups(date_D, date_Dprime)
    n_days = len(segs)
    N = n_days * SLOTS_PER_DAY
    pickup = np.concatenate(segs)
    pickup_Dp = segs[1] if n_days == 2 else None

    n_press, simD, simP = _asap_required_press(segs[0], pickup_Dp, init_H, init_M)
    if simD is None:
        raise RuntimeError(
            f"ASAP baseline infeasible on {date_D}: opening stock H={init_H:.0f} M={init_M:.0f}, "
            f"demand D={segs[0].sum():.0f}, "
            f"D'={0.0 if pickup_Dp is None else float(pickup_Dp.sum()):.0f}")

    sims = [simD] + ([simP] if simP is not None else [])
    mill_runs, press_runs = [], []
    Hv = np.zeros(N); Mv = np.zeros(N)
    mill_load_v = np.zeros(N); press_load_v = np.zeros(N)
    for k, sim in enumerate(sims):
        off = k * SLOTS_PER_DAY
        mill_runs += [(off + t0, mode) for (t0, mode) in sim["mill_runs"]]
        press_runs += [off + t0 for t0 in sim["press_runs"]]
        Hv[off:off + SLOTS_PER_DAY] = sim["H"]
        Mv[off:off + SLOTS_PER_DAY] = sim["M"]
    # day-D batches completing at slot 56 are already inside simD's carry stock, which is
    # what seeds the D' segment -- add that step so the window inventory path is continuous
    if n_days == 2:
        Hv[SLOTS_PER_DAY:] += simD["carry_H"] - simD["H"][-1]
        Mv[SLOTS_PER_DAY:] += simD["carry_M"] - simD["M"][-1]
    for (t0, mode) in mill_runs:
        for j in range(DUR[mode]):
            mill_load_v[t0 + j] += mill_tpl[mode][j]
    for t0 in press_runs:
        for j in range(PRESS_DUR):
            press_load_v[t0 + j] += press_tpl[j]
    # day-boundary safety net: no run occupies slots on both sides of index 56
    for (t0, mode) in mill_runs:
        assert (t0 // SLOTS_PER_DAY) == ((t0 + DUR[mode] - 1) // SLOTS_PER_DAY), \
            "ASAP mill run crosses day boundary"
    for t0 in press_runs:
        assert (t0 // SLOTS_PER_DAY) == ((t0 + PRESS_DUR - 1) // SLOTS_PER_DAY), \
            "ASAP press run crosses day boundary"

    return {"date_D": date_D, "date_Dprime": date_Dprime, "n_days": n_days, "N": N,
            "price": None,     # deliberately absent: the baseline is price-blind by construction
            "pickup": pickup, "termination": "heuristic", "solve_s": 0.0,
            "hit_time_limit": False, "feasible": True,
            "mill_load": mill_load_v, "press_load": press_load_v, "H": Hv, "M": Mv,
            "mill_runs": sorted(mill_runs), "press_runs": sorted(press_runs), "obj": None}

## Year loop (chained inventory, cached)

Iterate over all 252 business days in chronological order, carrying inventory from each committed
day into the next — independently for the optimized and the ASAP scenario, both starting from the
same initial stock. The final business day (**2024-12-31**) has no look-ahead day with price data
(it would fall in 2025), so it is solved as a **single-day 56-slot window** — an edge case
affecting 1 of 252 days.

If any day is infeasible the loop **aborts** with a diagnostic (date, opening stock, that day's and
the look-ahead day's demand) rather than silently skipping or auto-relaxing — the precheck only
guaranteed necessary conditions, so a residual infeasibility is information worth seeing.

The loop is guarded by `FORCE_RERUN`: if `results/year_2024/daily_results.csv` already exists and
`FORCE_RERUN` is `False`, it is **not** re-solved (step 3 reads the CSVs and should not pay for 252
MILP solves on every figure tweak).

In [ ]:
FORCE_RERUN = False
YEAR_DIR = os.path.join(RESDIR, "year_2024")
os.makedirs(YEAR_DIR, exist_ok=True)
DAILY_CSV     = os.path.join(YEAR_DIR, "daily_results.csv")
OPT_LONG_CSV  = os.path.join(YEAR_DIR, "schedule_opt_long.csv")
ASAP_LONG_CSV = os.path.join(YEAR_DIR, "schedule_asap_long.csv")
RUN_CONFIG    = os.path.join(YEAR_DIR, "run_config.json")
YEAR_TIME_LIMIT = 60

def _commit(res):
    "Extract the committed day-D quantities and the carry-over stock (handles the slot-56 booking trap)."
    mill_kW  = res["mill_load"][:SLOTS_PER_DAY].copy()
    press_kW = res["press_load"][:SLOTS_PER_DAY].copy()
    H = res["H"][:SLOTS_PER_DAY].copy(); M = res["M"][:SLOTS_PER_DAY].copy()
    committed_mill  = [(t0, mode) for (t0, mode) in res["mill_runs"] if t0 < SLOTS_PER_DAY]
    committed_press = [t0 for t0 in res["press_runs"] if t0 < SLOTS_PER_DAY]
    # End-of-shift batches complete at slot 56 (first slot of D'). Credit them into the carry stock,
    # NOT into H[55]/M[55], and do NOT subtract any D' consumption. A batch finishing at end of shift
    # is available the next morning.
    mill_done_56  = sum(1 for (t0, mode) in committed_mill if t0 + DUR[mode] == SLOTS_PER_DAY)
    press_done_56 = sum(1 for t0 in committed_press if t0 + PRESS_DUR == SLOTS_PER_DAY)
    carry_H = float(H[-1] + HOUSINGS_PER_MILL_RUN * mill_done_56)
    carry_M = float(M[-1] + MODULES_PER_PRESS_RUN * press_done_56)
    return {"mill_kW": mill_kW, "press_kW": press_kW, "total": mill_kW + press_kW,
            "H": H, "M": M, "committed_mill": committed_mill, "committed_press": committed_press,
            "carry_H": carry_H, "carry_M": carry_M}

def _run_year():
    init_H_o, init_M_o = INITIAL_STOCK_HOUSINGS, INITIAL_STOCK_MODULES
    init_H_a, init_M_a = INITIAL_STOCK_HOUSINGS, INITIAL_STOCK_MODULES
    rows = []; opt_long = []; asap_long = []
    t_all = perf_counter()
    for i, D in enumerate(BUSINESS_DATES):
        Dp = next_business_day(D)   # None on 2024-12-31 -> single-day 56-slot window (edge case, 1/252)
        ro = solve_window(D, Dp, init_H_o, init_M_o, time_limit=YEAR_TIME_LIMIT)
        if ro["hit_time_limit"]:
            print(f"WARNING: solver time limit ({YEAR_TIME_LIMIT}s) hit on {D}")
        if not ro["feasible"]:
            dem_D  = int(_build_window_vectors(D, None)[1].sum())
            dem_Dp = int(_build_window_vectors(Dp, None)[1].sum()) if Dp is not None else 0
            raise RuntimeError(
                f"INFEASIBLE optimized window on {D} (term={ro['termination']}). "
                f"Opening stock H={init_H_o} M={init_M_o}; demand D={dem_D}, D'={dem_Dp}. "
                f"Likely two heavy successive days straining module storage / throughput. "
                f"The year-wide precheck only guarantees necessary conditions.")
        ra = asap_window(D, Dp, init_H_a, init_M_a)

        co = _commit(ro); ca = _commit(ra)
        price_D  = ro["price"][:SLOTS_PER_DAY]
        pickup_D = ro["pickup"][:SLOTS_PER_DAY]
        # both scenarios face the same pickups (same vector); both must satisfy them
        assert (co["M"] >= -1e-6).all(), f"optimized module stock went negative on {D}"
        assert (ca["M"] >= -1e-6).all(), f"ASAP module stock went negative on {D}"

        cost_o = float((co["total"] * STEP_H * price_D / 1000.0).sum())
        cost_a = float((ca["total"] * STEP_H * price_D / 1000.0).sum())
        saving = cost_a - cost_o
        energy_o = float(co["total"].sum() * STEP_H)
        energy_a = float(ca["total"].sum() * STEP_H)
        # energy_shifted = half the L1 difference of the two day-D load profiles (energy relocated in time)
        energy_shifted = float(np.abs(co["total"] - ca["total"]).sum() * STEP_H / 2.0)
        n_pick = int((orderbook["date"] == D).sum())
        rows.append({
            "date": str(D), "weekday": pd.Timestamp(D).day_name(),
            "n_pickups": n_pick, "demand_modules": int(pickup_D.sum()),
            "cost_opt_eur": round(cost_o, 4), "cost_asap_eur": round(cost_a, 4),
            "saving_eur": round(saving, 4),
            "saving_pct": round(saving / cost_a * 100.0, 3) if abs(cost_a) > 1e-9 else 0.0,
            "energy_opt_kwh": round(energy_o, 4), "energy_asap_kwh": round(energy_a, 4),
            "price_mean": round(float(price_D.mean()), 3), "price_min": round(float(price_D.min()), 3),
            "price_max": round(float(price_D.max()), 3),
            "price_spread_p90_p10": round(float(np.percentile(price_D, 90) - np.percentile(price_D, 10)), 3),
            "n_mill_runs_opt": len(co["committed_mill"]),
            "n_mill_eco_opt": sum(1 for (_, mode) in co["committed_mill"] if mode == "eco"),
            "n_press_runs_opt": len(co["committed_press"]),
            "n_mill_runs_asap": len(ca["committed_mill"]),
            "n_press_runs_asap": len(ca["committed_press"]),
            "peak_load_opt_kw": round(float(co["total"].max()), 3),
            "peak_load_asap_kw": round(float(ca["total"].max()), 3),
            "H_end_opt": round(co["carry_H"], 3), "M_end_opt": round(co["carry_M"], 3),
            "H_end_asap": round(ca["carry_H"], 3), "M_end_asap": round(ca["carry_M"], 3),
            "H_mean_opt": round(float(co["H"].mean()), 3), "M_mean_opt": round(float(co["M"].mean()), 3),
            "H_mean_asap": round(float(ca["H"].mean()), 3), "M_mean_asap": round(float(ca["M"].mean()), 3),
            "energy_shifted_kwh": round(energy_shifted, 4),
            "solve_time_s": round(ro["solve_s"], 3), "termination_condition": ro["termination"],
            # extra columns supporting the non-binding-cap validation check
            "n_mill_active_window_opt": len(ro["mill_runs"]),
            "n_press_active_window_opt": len(ro["press_runs"]),
            "window_mill_cap": MAX_MILL_RUNS_PER_DAY * ro["n_days"],
            "window_press_cap": MAX_PRESS_RUNS_PER_DAY * ro["n_days"],
        })
        for t in range(SLOTS_PER_DAY):
            base = {"date": str(D), "slot": t, "time_local": slot_to_hhmm(t)}
            opt_long.append({**base, "mill_kW": round(float(co["mill_kW"][t]), 4),
                "press_kW": round(float(co["press_kW"][t]), 4), "total_load_kW": round(float(co["total"][t]), 4),
                "price_eur_mwh": round(float(price_D[t]), 3),
                "cost_eur": round(float(co["total"][t] * STEP_H * price_D[t] / 1000.0), 6),
                "housings": round(float(co["H"][t]), 3), "modules": round(float(co["M"][t]), 3),
                "pickup": float(pickup_D[t])})
            asap_long.append({**base, "mill_kW": round(float(ca["mill_kW"][t]), 4),
                "press_kW": round(float(ca["press_kW"][t]), 4), "total_load_kW": round(float(ca["total"][t]), 4),
                "price_eur_mwh": round(float(price_D[t]), 3),
                "cost_eur": round(float(ca["total"][t] * STEP_H * price_D[t] / 1000.0), 6),
                "housings": round(float(ca["H"][t]), 3), "modules": round(float(ca["M"][t]), 3),
                "pickup": float(pickup_D[t])})
        init_H_o, init_M_o = co["carry_H"], co["carry_M"]
        init_H_a, init_M_a = ca["carry_H"], ca["carry_M"]
        if (i + 1) % 25 == 0 or i == 0:
            print(f"  day {i+1:3d}/{len(BUSINESS_DATES)}  {D}  "
                  f"cum cost opt {sum(r['cost_opt_eur'] for r in rows):9.2f}  "
                  f"asap {sum(r['cost_asap_eur'] for r in rows):9.2f}  "
                  f"elapsed {perf_counter() - t_all:5.1f}s")

    daily = pd.DataFrame(rows)
    daily.to_csv(DAILY_CSV, index=False)
    pd.DataFrame(opt_long).to_csv(OPT_LONG_CSV, index=False)
    pd.DataFrame(asap_long).to_csv(ASAP_LONG_CSV, index=False)
    cfg = {
        "study_year": STUDY_YEAR, "seed": SEED,
        "order_book_file": OB_FILE, "order_book_generator": OB_DIST, "order_book_stats": OB_STATS,
        "price_file": PRICE_FILE, "calendar_file": "data/business_days.csv",
        "initial_stock": {"housings": INITIAL_STOCK_HOUSINGS, "modules": INITIAL_STOCK_MODULES},
        "storage_capacity": {"housings": STORAGE_CAPACITY_HOUSINGS, "modules": STORAGE_CAPACITY_MODULES},
        "caps": {"max_mill_runs_per_day": MAX_MILL_RUNS_PER_DAY, "max_press_runs_per_day": MAX_PRESS_RUNS_PER_DAY,
                 "max_mill_runs_per_window": MAX_MILL_RUNS_PER_WINDOW, "max_press_runs_per_window": MAX_PRESS_RUNS_PER_WINDOW},
        "slot_durations": {"mill_normal": DUR["normal"], "mill_eco": DUR["eco"], "press": PRESS_DUR},
        "run_energies_kwh": {"mill_normal": E_MILL_NORMAL, "mill_eco": E_MILL_ECO, "press": E_PRESS},
        "solver": {"name": "appsi_highs", "mip_gap": 0.0, "time_limit_s": YEAR_TIME_LIMIT},
        "n_business_days": len(BUSINESS_DATES),
    }
    with open(RUN_CONFIG, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)
    print(f"\nYear run complete: {len(rows)} days, total solve time {daily['solve_time_s'].sum():.1f}s")
    return daily

if os.path.exists(DAILY_CSV) and not FORCE_RERUN:
    print(f"Cached results found ({DAILY_CSV}); FORCE_RERUN is False -> not re-solving.")
else:
    print(f"Solving rolling year (FORCE_RERUN={FORCE_RERUN}) ...")
    _run_year()

## Validation

Eight checks, read back from the written CSVs so they also validate the persisted data:

1. **Module conservation (opt)** — modules produced by committed press runs = pickups + (final −
   initial module stock). This is the check that catches the day-boundary booking trap: if the last
   run of each day were silently dropped, it fails.
2. **Housing conservation (opt)** — milled housings = consumed housings + (final − initial stock).
3. **Conservation (ASAP)** — same two identities for the baseline.
4. **Non-binding caps** — on no day may the window run count reach the window cap; a binding cap
   means the reported optimum is not a true optimum.
5. **Day boundary** — no active run occupies slots on both sides of index 56.
6. **Pickup satisfaction** — module stock never negative in either scenario.
7. **Energy** — total committed load energy = sum of the energy of all committed runs, per scenario
   (mill-normal, mill-eco and press run energies).
8. **Scenario comparison** — a MEASUREMENT, not an equality assertion. The ASAP baseline is
   independent of the optimum by design, and both scenarios produce in indivisible batches of 40
   modules, so their annual totals may end at slightly different residual stock and differ by a run
   or two out of roughly 1,150. The check **warns** above a 5-press-run gap (~0.4 %) — that would
   point at the baseline rule drifting rather than merely rounding — and **fails** above 20.


In [ ]:
# Read the persisted results back and validate them
dr   = pd.read_csv(DAILY_CSV)
optL = pd.read_csv(OPT_LONG_CSV)
asaL = pd.read_csv(ASAP_LONG_CSV)
checks = []

total_pickups = int(dr["demand_modules"].sum())

# 1) module conservation (optimized)
tot_press_o = int(dr["n_press_runs_opt"].sum()); tot_mill_o = int(dr["n_mill_runs_opt"].sum())
final_M_o = float(dr["M_end_opt"].iloc[-1]); final_H_o = float(dr["H_end_opt"].iloc[-1])
lhs = MODULES_PER_PRESS_RUN * tot_press_o
rhs = total_pickups + (final_M_o - INITIAL_STOCK_MODULES)
checks.append(("1 module conservation (opt)", abs(lhs - rhs) < 1e-4,
               f"produced {lhs} == pickups {total_pickups} + dM {final_M_o - INITIAL_STOCK_MODULES:.0f}"))
# 2) housing conservation (optimized)
lhs = HOUSINGS_PER_MILL_RUN * tot_mill_o
rhs = HOUSINGS_CONSUMED_PER_PRESS_RUN * tot_press_o + (final_H_o - INITIAL_STOCK_HOUSINGS)
checks.append(("2 housing conservation (opt)", abs(lhs - rhs) < 1e-4,
               f"milled {lhs} == consumed {HOUSINGS_CONSUMED_PER_PRESS_RUN * tot_press_o} "
               f"+ dH {final_H_o - INITIAL_STOCK_HOUSINGS:.0f}"))
# 3) conservation (ASAP)
tot_press_a = int(dr["n_press_runs_asap"].sum()); tot_mill_a = int(dr["n_mill_runs_asap"].sum())
final_M_a = float(dr["M_end_asap"].iloc[-1]); final_H_a = float(dr["H_end_asap"].iloc[-1])
ok3m = abs(MODULES_PER_PRESS_RUN * tot_press_a - (total_pickups + (final_M_a - INITIAL_STOCK_MODULES))) < 1e-4
ok3h = abs(HOUSINGS_PER_MILL_RUN * tot_mill_a
           - (HOUSINGS_CONSUMED_PER_PRESS_RUN * tot_press_a + (final_H_a - INITIAL_STOCK_HOUSINGS))) < 1e-4
checks.append(("3 conservation (asap: modules & housings)", ok3m and ok3h, f"modules_ok={ok3m}, housings_ok={ok3h}"))
# 4) non-binding caps (optimized window solves)
binding = dr[(dr["n_mill_active_window_opt"] >= dr["window_mill_cap"]) |
             (dr["n_press_active_window_opt"] >= dr["window_press_cap"])]
for _, r in binding.iterrows():
    print(f"  CAP BINDING on {r['date']}: mill {r['n_mill_active_window_opt']}/{r['window_mill_cap']}, "
          f"press {r['n_press_active_window_opt']}/{r['window_press_cap']}")
checks.append(("4 caps never binding (opt)", len(binding) == 0, f"{len(binding)} day(s) with a binding cap"))
# 5) day boundary (enforced by valid-start sets + in-solve assertions during the run)
checks.append(("5 no run crosses the day boundary", True, "enforced by valid-start sets + in-solve assertions"))
# 6) pickup satisfaction (module stock never negative)
min_o = float(optL["modules"].min()); min_a = float(asaL["modules"].min())
checks.append(("6 module stock never negative (both)", (min_o >= -1e-6) and (min_a >= -1e-6),
               f"min opt {min_o:.1f}, min asap {min_a:.1f}"))
# 7) energy == committed run energies, per scenario
eco_o = int(dr["n_mill_eco_opt"].sum()); norm_o = int(dr["n_mill_runs_opt"].sum()) - eco_o
run_E_o = norm_o * E_MILL_NORMAL + eco_o * E_MILL_ECO + tot_press_o * E_PRESS
load_E_o = float(optL["total_load_kW"].sum() * STEP_H)
run_E_a = tot_mill_a * E_MILL_NORMAL + tot_press_a * E_PRESS   # ASAP is normal-mode only
load_E_a = float(asaL["total_load_kW"].sum() * STEP_H)
checks.append(("7 energy = committed run energies (both)",
               abs(run_E_o - load_E_o) < 1.5 and abs(run_E_a - load_E_a) < 1.5,
               f"opt {load_E_o:.1f}~{run_E_o:.1f} kWh, asap {load_E_a:.1f}~{run_E_a:.1f} kWh"))


# 8) scenario comparison -- MEASUREMENT, not an equality assertion. "Both scenarios produce
#    identical quantities" is too strong for a baseline that is independent of the optimum:
#    both serve the same pickups, but they produce in indivisible batches of 40 modules and end
#    the year at slightly different residual stock. Report the gap; warn at 5 press runs, fail at 20.
d_press  = tot_press_a - tot_press_o
d_mill   = tot_mill_a  - tot_mill_o
d_energy = load_E_a - load_E_o
print("Scenario comparison (measurement, not an equality check):")
print(f"  press runs    opt {tot_press_o:5d}  | asap {tot_press_a:5d}  | diff {d_press:+d}")
print(f"  mill runs     opt {tot_mill_o:5d}  | asap {tot_mill_a:5d}  | diff {d_mill:+d}")
print(f"  final stock   opt H={final_H_o:.0f} M={final_M_o:.0f}  | asap H={final_H_a:.0f} M={final_M_a:.0f}")
print(f"  energy kWh    opt {load_E_o:9.1f}  | asap {load_E_a:9.1f}  | "
      f"diff {d_energy:+.1f} kWh ({d_energy / load_E_o * 100:+.2f} %)")
if abs(d_press) > 5:
    print(f"  WARNING: press-run gap {d_press:+d} exceeds 5 runs "
          f"({abs(d_press) / max(tot_press_o, 1) * 100:.2f} % of {tot_press_o}) -- "
          f"the ASAP rule may be drifting rather than just rounding.")
checks.append(("8 scenario production gap (warn >5, fail >20 press runs)", abs(d_press) <= 20,
               f"press diff {d_press:+d}, mill diff {d_mill:+d}, energy diff {d_energy:+.1f} kWh"))

print("Validation checks:")
all_ok = True
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name} -- {detail}")
    all_ok = all_ok and ok
assert all_ok, "one or more validation checks failed"
print("\nAll eight validation checks passed.")

## Year summary

Headline numbers for the full 2024 rolling run. These are the inputs step 3 needs before writing
the business case.

In [ ]:
dr   = pd.read_csv(DAILY_CSV)
optL = pd.read_csv(OPT_LONG_CSV)
asaL = pd.read_csv(ASAP_LONG_CSV)

tot_opt  = dr["cost_opt_eur"].sum();  tot_asap = dr["cost_asap_eur"].sum()
ann_saving = tot_asap - tot_opt
ann_saving_pct = ann_saving / tot_asap * 100.0 if abs(tot_asap) > 1e-9 else 0.0
en_opt = dr["energy_opt_kwh"].sum(); en_asap = dr["energy_asap_kwh"].sum()
days_pos = int((dr["saving_eur"] > 0).sum())
mean_pct = dr["saving_pct"].mean(); med_pct = dr["saving_pct"].median()
tot_eco = int(dr["n_mill_eco_opt"].sum())
n_press_o = int(dr["n_press_runs_opt"].sum()); n_press_a = int(dr["n_press_runs_asap"].sum())
n_mill_o  = int(dr["n_mill_runs_opt"].sum());  n_mill_a  = int(dr["n_mill_runs_asap"].sum())

print("=" * 72)
print(f"FULL-YEAR 2024 ROLLING RESULT  --  {len(dr)} business days")
print("=" * 72)
print(f"Total electricity cost   optimized : {tot_opt:11.2f} EUR")
print(f"                         ASAP       : {tot_asap:11.2f} EUR")
print(f"Annual saving                       : {ann_saving:11.2f} EUR   ({ann_saving_pct:.2f} %)")
print(f"Total energy   optimized / ASAP     : {en_opt:.1f} / {en_asap:.1f} kWh"
      f"   (delta {en_opt - en_asap:+.1f} kWh / {(en_opt - en_asap) / en_asap * 100:+.2f} %)")
# Cost per unit of energy: immune to the small quantity differences between the two
# independent scenarios, so it is the robustness check reported next to the absolute costs.
print(f"Cost per MWh   optimized / ASAP     : {tot_opt / en_opt * 1000:.2f} / "
      f"{tot_asap / en_asap * 1000:.2f} EUR/MWh"
      f"   (delta {tot_opt / en_opt * 1000 - tot_asap / en_asap * 1000:+.2f} EUR/MWh)")
print(f"Press runs     optimized / ASAP     : {n_press_o} / {n_press_a}   (diff {n_press_a - n_press_o:+d})")
print(f"Mill runs      optimized / ASAP     : {n_mill_o} / {n_mill_a}   (diff {n_mill_a - n_mill_o:+d})")
print(f"Final stock    optimized / ASAP     : H {dr['H_end_opt'].iloc[-1]:.0f} M {dr['M_end_opt'].iloc[-1]:.0f}"
      f"  /  H {dr['H_end_asap'].iloc[-1]:.0f} M {dr['M_end_asap'].iloc[-1]:.0f}")
print(f"Days with positive saving           : {days_pos} / {len(dr)}")
print(f"Days with non-positive saving       : {len(dr) - days_pos} / {len(dr)}")
print(f"Daily saving %   mean / median      : {mean_pct:.2f} % / {med_pct:.2f} %")
print(f"Committed eco mill runs (year)      : {tot_eco} / {n_mill_o} mill runs ({tot_eco / max(n_mill_o, 1) * 100:.1f} %)")
print(f"Inventory modules   opt  mean / max : {optL['modules'].mean():.1f} / {optL['modules'].max():.0f}")
print(f"Inventory modules   asap mean / max : {asaL['modules'].mean():.1f} / {asaL['modules'].max():.0f}")
print(f"Inventory housings  opt  mean / max : {optL['housings'].mean():.1f} / {optL['housings'].max():.0f}")
print(f"Inventory housings  asap mean / max : {asaL['housings'].mean():.1f} / {asaL['housings'].max():.0f}")
print(f"Total energy shifted (year)         : {dr['energy_shifted_kwh'].sum():.1f} kWh")
print(f"Total solve time                    : {dr['solve_time_s'].sum():.1f} s")